# Causal Swing Structure Research Notebook

Strictly causal rule: every state shown for row `t` uses only information available up to `t-1`.

## 1. Setup and imports

In [ ]:
from __future__ import annotations
from collections import defaultdict
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display


## 2. Configuration

In [ ]:
DATA_FILE_PATH=r".\data\data.parquet"
SELECTED_TICKER=None
ANALYSIS_START_DATE="2014-01-01"
ANALYSIS_END_DATE=None
ROLLING_LOOKBACK=200
ATR_WINDOW=14
ATR_MULTIPLIER=1.5
MIN_BARS_CONFIRMATION=3
PCT_THRESHOLD=0.02
CONFIRMATION_MODE="atr"
CONFIRMATION_PRICE_MODE="hybrid"  # close, wick, hybrid
TRADING_DAYS_1W=5
TRADING_DAYS_1M=21
TRADING_DAYS_3M=63
TRADING_DAYS_6M=126
TRADING_DAYS_1Y=252
CALENDAR_EQ_WINDOWS=[1,TRADING_DAYS_1W,TRADING_DAYS_1M,TRADING_DAYS_3M,TRADING_DAYS_6M,TRADING_DAYS_1Y]
EXTRA_STRUCTURE_WINDOWS=[3,10,300]
LEVEL_WINDOWS=sorted(set(CALENDAR_EQ_WINDOWS+EXTRA_STRUCTURE_WINDOWS))
LEVEL_CONFLUENCE_REQUIRED=2
LEVEL_TOLERANCE_ATR_MULTIPLIER=0.75
REQUIRE_MULTI_HORIZON_CONFLUENCE=True
SHORT_CONFLUENCE_MAX_WINDOW=20
LONG_CONFLUENCE_MIN_WINDOW=60
MIN_SHORT_CONFLUENCE_HITS=1
MIN_LONG_CONFLUENCE_HITS=1
MAX_CANDIDATE_BARS=18
FORCE_CONFIRMATION_AFTER_BARS=7
PLOT_LAST_N_BARS=1250
STRUCTURE_BREAK_SWITCH_ATR_MULTIPLIER=0.35
STRUCTURE_BREAK_SWITCH_PCT=0.003
LOCAL_SWING_WINDOW=20
LOCAL_SWING_MIN_REACTION_MULTIPLIER=1.05
CANDIDATE_REPLACE_MIN_ATR_STEP=0.10
CANDIDATE_REPLACE_MIN_PCT_STEP=0.001
PLOT_CANDIDATE_MODE="significant"  # all, significant, break_only
PLOT_MIN_CONFLUENCE_FOR_CANDIDATE=3
PLOT_INCLUDE_SEEDED_CANDIDATES=False
PLOT_SIGNIFICANT_MIN_GAP_DAYS=12
PLOT_CANDIDATE_MARKER_SIZE=8
PLOT_CANDIDATE_MARKER_OPACITY=0.72
PLOT_BREAK_ATTEMPT_MODE="off"  # all, significant, off
PLOT_BREAK_MIN_CONFLUENCE=3
PLOT_BREAK_MIN_GAP_DAYS=8
PLOT_CONFIRMED_STYLE_MODE="tiered"  # basic, tiered
PLOT_CONFIRMED_TIER_FILTER="all"  # all, structural_plus, global_only
PLOT_CONFIRMED_MIN_GAP_DAYS_LOCAL=6
PLOT_CONFIRMED_MIN_GAP_DAYS_STRUCTURAL=6
PLOT_CONFIRMED_MIN_GAP_DAYS_GLOBAL=3
PLOT_STRUCTURAL_MIN_TOTAL_CONFLUENCE=3
PLOT_STRUCTURAL_MIN_LONG_HITS=1
PLOT_GLOBAL_MIN_TOTAL_CONFLUENCE=4
PLOT_GLOBAL_MIN_LONG_HITS=2
PLOT_GLOBAL_REQUIRE_200_WINDOW=False
MICRO_STRUCTURE_EVENT_LOOKBACK=9
MICRO_STRUCTURE_MIN_CONFLUENCE=3
MICRO_STRUCTURE_TIER_MODE="structural_plus"  # all, local_plus, structural_plus
FIB_RATIOS=[0.0,0.236,0.382,0.5,0.618,0.786,1.0]
FIB_ANCHOR_TIER_FILTER="structural_plus"  # all, structural_plus, global_only
FIB_ANCHOR_METHOD="hh_ll_confluence"  # latest_opposite, hh_ll_confluence
FIB_HH_LL_LOOKBACK_EVENTS=120
FIB_MIN_SWING_RANGE_ATR=0.8


## 3. Load and inspect data

In [ ]:
def load_data(fp):
    path=Path(fp)
    if not path.exists(): raise FileNotFoundError(path)
    suf=path.suffix.lower()
    if suf==".parquet": return pd.read_parquet(path)
    if suf==".csv": return pd.read_csv(path)
    if suf==".feather": return pd.read_feather(path)
    if suf in {".pkl",".pickle"}: return pd.read_pickle(path)
    raise ValueError(f"Unsupported file type: {suf}")
raw_df=load_data(DATA_FILE_PATH)
print("path",Path(DATA_FILE_PATH).resolve())
print("shape",raw_df.shape)
print("index",type(raw_df.index).__name__,raw_df.index.name)
display(raw_df.dtypes.head(25).to_frame("dtype"))
display(raw_df.head())
display(pd.DataFrame({"column":raw_df.columns.tolist()}))


## 4. Automatic OHLC column detection

In [ ]:
def norm(x): return str(x).strip().lower()
DATE={"date","datetime","timestamp","time","dt","trading_date"}
OPEN={"open","o"}; HIGH={"high","h"}; LOW={"low","l"}
CLOSE={"close","c","price","adj_close","adjusted_close","settle","last"}
VOL={"volume","vol","qty","turnover"}; TICK={"ticker","symbol","asset","instrument","security","contract"}
def score(name,aliases):
    n=norm(name).replace(" ","_"); s=100 if n in aliases else 0
    for a in aliases:
        if n==a: s=max(s,90)
        if n.startswith(a+"_") or n.endswith("_"+a): s=max(s,35)
        if a in n: s=max(s,10)
    return s
def choose(df,aliases,label,required=True):
    xs=sorted([(score(col,aliases),str(col)) for col in df.columns if score(col,aliases)>0], key=lambda z:(-z[0],z[1]))
    if not xs:
        if required: raise ValueError(f"Could not detect {label}. Available columns: {list(df.columns)}")
        return None
    top=xs[0][0]; best=[col for sc,col in xs if sc==top]
    if len(best)>1: raise ValueError(f"Ambiguous {label} candidates: {best}")
    return best[0]
def detect_date(df):
    if isinstance(df.index,pd.DatetimeIndex): return {"source":"index","column":df.index.name or "index"}
    return {"source":"column","column":choose(df,DATE,"date",True)}
def detect_wide(df):
    grouped=defaultdict(dict); alias_map={"open":OPEN,"high":HIGH,"low":LOW,"close":CLOSE,"volume":VOL}
    for col in df.columns:
        n=norm(col); matched=False
        for field,aliases in alias_map.items():
            for a in sorted(aliases,key=len,reverse=True):
                suf=f"_{a}"
                if n.endswith(suf):
                    ticker=str(col)[:-len(suf)]
                    if ticker: grouped[ticker][field]=str(col)
                    matched=True; break
            if matched: break
    if not grouped: return None
    full=[t for t,m in grouped.items() if {"open","high","low","close"}.issubset(m)]
    if not full: return None
    ordered=[]
    for col in df.columns:
        for t in full:
            if str(col).startswith(f"{t}_") and t not in ordered: ordered.append(t)
    sel=SELECTED_TICKER or ordered[0]
    if sel not in grouped: raise ValueError(f"SELECTED_TICKER not found: {sel}. Detected groups: {sorted(grouped)}")
    if sel not in full: raise ValueError(f"Ticker {sel} does not have full OHLC fields: {grouped[sel]}")
    return {"mode":"wide","date_source":detect_date(df),"selected_ticker":sel,"wide_groups":dict(grouped),"full_ohlc_tickers":ordered,"selected_mapping":grouped[sel]}
def detect_tidy(df):
    d=detect_date(df); m={"date":d["column"],"open":choose(df,OPEN,"open",True),"high":choose(df,HIGH,"high",True),"low":choose(df,LOW,"low",True),"close":choose(df,CLOSE,"close",True),"volume":choose(df,VOL,"volume",False),"ticker":choose(df,TICK,"ticker",False)}
    return {"mode":"tidy","date_source":d,"selected_ticker":SELECTED_TICKER,"selected_mapping":m}
def detect_schema(df):
    w=detect_wide(df)
    return w if w is not None else detect_tidy(df)
schema_info=detect_schema(raw_df)
print("mode",schema_info["mode"])
print("date source",schema_info["date_source"])
if schema_info["mode"]=="wide":
    display(pd.DataFrame({"ticker":schema_info["full_ohlc_tickers"]}))
    partial=[{"ticker":t,"fields":sorted(m)} for t,m in schema_info["wide_groups"].items() if t not in schema_info["full_ohlc_tickers"]]
    if partial: display(pd.DataFrame(partial).sort_values("ticker").reset_index(drop=True))
    print("selected ticker",schema_info["selected_ticker"])
    display(pd.DataFrame([schema_info["selected_mapping"]]))
else:
    display(pd.DataFrame([schema_info["selected_mapping"]]))


## 5. Data preparation

In [ ]:
def prepare(df,schema):
    ds=schema["date_source"]
    if schema["mode"]=="wide":
        t=schema["selected_ticker"]; m=schema["selected_mapping"]
        out=pd.DataFrame({"date":pd.to_datetime(df.index if ds["source"]=="index" else df[ds["column"]]).to_numpy(),"ticker":t,"open":pd.to_numeric(df[m["open"]],errors="coerce").to_numpy(),"high":pd.to_numeric(df[m["high"]],errors="coerce").to_numpy(),"low":pd.to_numeric(df[m["low"]],errors="coerce").to_numpy(),"close":pd.to_numeric(df[m["close"]],errors="coerce").to_numpy()})
        out["volume"]=pd.to_numeric(df[m["volume"]],errors="coerce").to_numpy() if "volume" in m else np.nan
    else:
        m=schema["selected_mapping"]
        out=pd.DataFrame({"date":pd.to_datetime(df.index if ds["source"]=="index" else df[m["date"]]).to_numpy(),"ticker":"SINGLE_ASSET","open":pd.to_numeric(df[m["open"]],errors="coerce").to_numpy(),"high":pd.to_numeric(df[m["high"]],errors="coerce").to_numpy(),"low":pd.to_numeric(df[m["low"]],errors="coerce").to_numpy(),"close":pd.to_numeric(df[m["close"]],errors="coerce").to_numpy()})
        out["volume"]=pd.to_numeric(df[m["volume"]],errors="coerce").to_numpy() if m.get("volume") else np.nan
        if m.get("ticker"): out["ticker"]=df[m["ticker"]].astype(str)
        if SELECTED_TICKER is not None: out=out[out["ticker"]==SELECTED_TICKER].copy()
    start_dt=None if ANALYSIS_START_DATE in [None,"","None"] else pd.Timestamp(ANALYSIS_START_DATE)
    end_dt=None if ANALYSIS_END_DATE in [None,"","None"] else pd.Timestamp(ANALYSIS_END_DATE)
    if start_dt is not None: out=out[out["date"]>=start_dt].copy()
    if end_dt is not None: out=out[out["date"]<=end_dt].copy()
    out=out.dropna(subset=["date","open","high","low","close"]).sort_values(["ticker","date"]).drop_duplicates(["ticker","date"],keep="last").reset_index(drop=True)
    if out.empty: raise ValueError(f"No rows left after date filters. ANALYSIS_START_DATE={ANALYSIS_START_DATE}, ANALYSIS_END_DATE={ANALYSIS_END_DATE}")
    if (out["high"]<out["low"]).any(): raise ValueError("Found rows with high < low.")
    return out
price_df=prepare(raw_df,schema_info)
print("prepared shape",price_df.shape)
print("tickers",sorted(price_df["ticker"].unique().tolist()))
print("analysis filter", ANALYSIS_START_DATE, ANALYSIS_END_DATE)
print("date range",price_df["date"].min(),price_df["date"].max())
if ANALYSIS_START_DATE not in [None,"","None"] and pd.Timestamp(price_df["date"].min()) > pd.Timestamp(ANALYSIS_START_DATE):
    print(f"note: requested start {ANALYSIS_START_DATE} but available data starts at {price_df['date'].min().date()}")
display(price_df.head(10))


## 6. ATR helper, rolling levels, and utility functions

In [ ]:
def causal_atr(df, window=14):
    pc = df["close"].shift(1)
    tr = pd.concat([(df["high"] - df["low"]), (df["high"] - pc).abs(), (df["low"] - pc).abs()], axis=1).max(axis=1)
    return tr.rolling(window=window, min_periods=1).mean()


def conf_dist(v, atr, mode="atr", atr_mult=1.5, pct=0.02):
    pdist = abs(v) * pct
    adist = np.nan if pd.isna(atr) else atr_mult * atr
    mode = mode.lower()
    if mode == "atr":
        return pdist if pd.isna(adist) else adist
    if mode == "pct":
        return pdist
    if mode == "max":
        return pdist if pd.isna(adist) else max(adist, pdist)
    if mode == "min":
        return pdist if pd.isna(adist) else min(adist, pdist)
    raise ValueError(f"Unsupported CONFIRMATION_MODE={mode}")


def rolling_slice(df, end_idx, lookback):
    start_idx = max(0, end_idx - lookback + 1)
    return df.iloc[start_idx : end_idx + 1]


def build_rolling_levels(df, windows):
    out = df.copy()
    for w in windows:
        # Allow very short windows (1/2/3 days) safely while keeping stricter warm-up for long windows.
        min_periods = max(1, min(int(w), max(5, int(w) // 4)))
        out[f"rolling_high_{w}"] = out["high"].shift(1).rolling(window=w, min_periods=min_periods).max()
        out[f"rolling_low_{w}"] = out["low"].shift(1).rolling(window=w, min_periods=min_periods).min()
    return out


def candidate_confluence(kind, value, row, windows, atr_value, tolerance_atr_multiplier):
    tol = 0.0 if pd.isna(atr_value) else float(atr_value) * float(tolerance_atr_multiplier)
    matches = []
    for w in windows:
        level_col = f"rolling_high_{w}" if kind == "high" else f"rolling_low_{w}"
        level = row.get(level_col, np.nan)
        if pd.isna(level):
            continue
        if kind == "high" and float(value) >= float(level) - tol:
            matches.append(int(w))
        if kind == "low" and float(value) <= float(level) + tol:
            matches.append(int(w))
    return matches


def excursion_metrics(kind, candidate_value, row):
    if kind == "high":
        wick_exc = float(candidate_value - row["low"])
        close_exc = float(candidate_value - row["close"])
    else:
        wick_exc = float(row["high"] - candidate_value)
        close_exc = float(row["close"] - candidate_value)
    return {"wick": wick_exc, "close": close_exc}


## 7. Candidate swing detection logic

Updated candidate logic:
- structure alternates between searching for highs and lows
- candidate wick extremes keep refining until confirmation
- rolling multi-horizon high-low context (5/10/20/60/100/200) is tracked as confluence
- break attempts are recorded as fresh events instead of staying noisy for many bars


In [ ]:
def empty_cand(kind):
    return {"kind": kind, "value": None, "date": pd.NaT, "index": None, "bars_active": 0, "atr_at_candidate": np.nan, "breaks_structure": False, "reference_confirmed_value": None, "seed_reason": None, "confluence_windows": [], "confluence_count": 0, "confluence_short_hits": 0, "confluence_long_hits": 0}


def confluence_bucket_hits(confluence_windows, short_max_window=20, long_min_window=60):
    ws = [int(w) for w in confluence_windows]
    short_hits = sum(1 for w in ws if w <= int(short_max_window))
    long_hits = sum(1 for w in ws if w >= int(long_min_window))
    return short_hits, long_hits


def refresh_candidate(kind, candidate, latest_confirmed, row, windows, tolerance_atr_multiplier):
    candidate = dict(candidate)
    ref_value = latest_confirmed.get("value")
    candidate["reference_confirmed_value"] = ref_value
    candidate["confluence_windows"] = candidate_confluence(kind, candidate["value"], row, windows, candidate["atr_at_candidate"], tolerance_atr_multiplier)
    candidate["confluence_count"] = len(candidate["confluence_windows"])
    short_max = int(globals().get("SHORT_CONFLUENCE_MAX_WINDOW", 20))
    long_min = int(globals().get("LONG_CONFLUENCE_MIN_WINDOW", 60))
    short_hits, long_hits = confluence_bucket_hits(candidate["confluence_windows"], short_max, long_min)
    candidate["confluence_short_hits"] = int(short_hits)
    candidate["confluence_long_hits"] = int(long_hits)
    if candidate["index"] is None or ref_value is None:
        candidate["breaks_structure"] = False
    else:
        candidate["breaks_structure"] = candidate["value"] > ref_value if kind == "high" else candidate["value"] < ref_value
    return candidate


def build_candidate(kind, row, i, atr, latest_confirmed, reason, windows, tolerance_atr_multiplier):
    field = "high" if kind == "high" else "low"
    candidate = {"kind": kind, "value": float(row[field]), "date": pd.Timestamp(row["date"]), "index": int(i), "bars_active": 0, "atr_at_candidate": atr, "breaks_structure": False, "reference_confirmed_value": None, "seed_reason": reason, "confluence_windows": [], "confluence_count": 0, "confluence_short_hits": 0, "confluence_long_hits": 0}
    return refresh_candidate(kind, candidate, latest_confirmed, row, windows, tolerance_atr_multiplier)


def build_candidate_from_window(kind, window, latest_confirmed, reason, windows, tolerance_atr_multiplier):
    field = "high" if kind == "high" else "low"
    idx = window[field].idxmax() if kind == "high" else window[field].idxmin()
    row = window.loc[idx]
    atr = float(row["atr"]) if not pd.isna(row["atr"]) else np.nan
    return build_candidate(kind, row, int(idx), atr, latest_confirmed, reason, windows, tolerance_atr_multiplier)


def candidate_is_stale(candidate, current_index, lookback, max_candidate_bars):
    if candidate["index"] is None:
        return False
    return (current_index - candidate["index"]) >= lookback or candidate["bars_active"] >= max_candidate_bars


def push_candidate_event(stats, kind, candidate, available_on, event_type, reason):
    stats[f"candidate_{kind}_events"].append({"event_type": event_type, "available_on": pd.Timestamp(available_on), "swing_date": pd.Timestamp(candidate["date"]), "value": float(candidate["value"]), "reason": reason, "breaks_structure": bool(candidate["breaks_structure"]), "reference_confirmed_value": candidate["reference_confirmed_value"], "confluence_count": int(candidate["confluence_count"]), "confluence_short_hits": int(candidate.get("confluence_short_hits", 0)), "confluence_long_hits": int(candidate.get("confluence_long_hits", 0)), "confluence_windows": ",".join(map(str, candidate["confluence_windows"]))})


def push_break_attempt_event(stats, kind, candidate, available_on):
    stats[f"break_attempt_{kind}_events"].append({"available_on": pd.Timestamp(available_on), "swing_date": pd.Timestamp(candidate["date"]), "value": float(candidate["value"]), "reference_confirmed_value": candidate["reference_confirmed_value"], "confluence_count": int(candidate["confluence_count"]), "confluence_short_hits": int(candidate.get("confluence_short_hits", 0)), "confluence_long_hits": int(candidate.get("confluence_long_hits", 0)), "confluence_windows": ",".join(map(str, candidate["confluence_windows"])), "seed_reason": candidate["seed_reason"]})


def should_replace_candidate(kind, new_value, old_value, atr_value, min_atr_step=0.15, min_pct_step=0.001):
    if kind == "high" and not (float(new_value) > float(old_value)):
        return False
    if kind == "low" and not (float(new_value) < float(old_value)):
        return False
    abs_delta = abs(float(new_value) - float(old_value))
    atr_step = 0.0 if pd.isna(atr_value) else float(atr_value) * float(min_atr_step)
    pct_step = abs(float(old_value)) * float(min_pct_step)
    min_required_delta = max(atr_step, pct_step)
    return abs_delta >= min_required_delta


def update_active_candidate(kind, state, row, i, atr, work, lookback, stats, windows, tolerance_atr_multiplier, max_candidate_bars, candidate_replace_min_atr_step=0.15, candidate_replace_min_pct_step=0.001):
    key = f"candidate_{kind}"
    latest_key = f"latest_confirmed_{kind}"
    replace_key = f"candidate_{kind}_replaced_before_confirmation"
    field = "high" if kind == "high" else "low"
    candidate = state[key]
    latest_confirmed = state[latest_key]
    prev_break = bool(candidate.get("breaks_structure", False)) if candidate["index"] is not None else False
    changed = False
    new_break = False
    if candidate["index"] is None:
        state[key] = build_candidate(kind, row, i, atr, latest_confirmed, "active_side_seed", windows, tolerance_atr_multiplier)
        push_candidate_event(stats, kind, state[key], row["date"], f"candidate_{kind}_seeded", "active_side_seed")
        new_break = bool(state[key]["breaks_structure"])
        if new_break:
            push_break_attempt_event(stats, kind, state[key], row["date"])
        return state, {"changed": True, "new_break": new_break}
    state[key]["bars_active"] += 1
    if should_replace_candidate(kind, row[field], candidate["value"], atr, candidate_replace_min_atr_step, candidate_replace_min_pct_step):
        stats[replace_key] += 1
        state[key] = build_candidate(kind, row, i, atr, latest_confirmed, "better_extreme_wick", windows, tolerance_atr_multiplier)
        push_candidate_event(stats, kind, state[key], row["date"], f"candidate_{kind}_replaced", "better_extreme_wick")
        changed = True
    elif candidate_is_stale(candidate, i, lookback, max_candidate_bars):
        replacement = build_candidate_from_window(kind, rolling_slice(work, i, lookback), latest_confirmed, "lookback_reanchor", windows, tolerance_atr_multiplier)
        if replacement["index"] != candidate["index"] or replacement["value"] != candidate["value"]:
            stats[replace_key] += 1
            state[key] = replacement
            push_candidate_event(stats, kind, state[key], row["date"], f"candidate_{kind}_reanchored", "lookback_reanchor")
            changed = True
        else:
            state[key] = refresh_candidate(kind, state[key], latest_confirmed, row, windows, tolerance_atr_multiplier)
    else:
        state[key] = refresh_candidate(kind, state[key], latest_confirmed, row, windows, tolerance_atr_multiplier)
    current_break = bool(state[key].get("breaks_structure", False))
    new_break = current_break and (changed or not prev_break)
    if new_break:
        push_break_attempt_event(stats, kind, state[key], row["date"])
    return state, {"changed": changed, "new_break": new_break}


## 8. Confirmed swing detection logic

Updated confirmation logic:
- reversal can use close, wick, or hybrid mode
- multi-horizon confluence (short + long windows) is required for fast confirmation
- low-confluence candidates can still confirm, but only after surviving longer
- wick extremes remain valid because the active candidate keeps refining before confirmation


In [ ]:
def structure_label(kind, candidate_value, previous_same_side_value):
    if previous_same_side_value is None:
        return "INITIAL_HIGH" if kind == "high" else "INITIAL_LOW"
    if kind == "high":
        return "HH" if candidate_value > previous_same_side_value else "LH"
    return "HL" if candidate_value > previous_same_side_value else "LL"


def confirmation_signal(kind, row, candidate_value, threshold, price_mode, confluence_count, confluence_required, bars_since_candidate, min_bars):
    metrics = excursion_metrics(kind, candidate_value, row)
    wick_exc = metrics["wick"]
    close_exc = metrics["close"]
    price_mode = price_mode.lower()
    if price_mode == "wick":
        passed = wick_exc >= threshold
        mode_used = "wick"
    elif price_mode == "close":
        passed = close_exc >= threshold
        mode_used = "close"
    elif price_mode == "hybrid":
        passed = close_exc >= threshold
        mode_used = "close"
        wick_override = confluence_count >= confluence_required and bars_since_candidate >= (min_bars + 1) and wick_exc >= (threshold * 1.25)
        if not passed and wick_override:
            passed = True
            mode_used = "wick_override"
    else:
        raise ValueError(f"Unsupported CONFIRMATION_PRICE_MODE={price_mode}")
    return passed, mode_used, metrics


def peek_confirmation(kind, state, row, i, atr, min_bars, mode, atr_mult, pct, confluence_required, force_confirmation_after_bars, confirmation_price_mode, local_swing_window, local_swing_min_reaction_multiplier, require_multi_horizon_confluence=False, min_short_confluence_hits=1, min_long_confluence_hits=1):
    candidate = state[f"candidate_{kind}"]
    latest_confirmed = state[f"latest_confirmed_{kind}"]
    if candidate["index"] is None:
        return None
    bars_since_candidate = i - candidate["index"]
    if bars_since_candidate < min_bars:
        return None
    threshold = conf_dist(float(candidate["value"]), atr if not pd.isna(atr) else candidate["atr_at_candidate"], mode, atr_mult, pct)
    signal_passed, mode_used, metrics = confirmation_signal(kind, row, float(candidate["value"]), float(threshold), confirmation_price_mode, int(candidate["confluence_count"]), int(confluence_required), int(bars_since_candidate), int(min_bars))
    if not signal_passed:
        return None
    short_hits = int(candidate.get("confluence_short_hits", 0))
    long_hits = int(candidate.get("confluence_long_hits", 0))
    multi_horizon_ok = True
    if bool(require_multi_horizon_confluence):
        multi_horizon_ok = short_hits >= int(min_short_confluence_hits) and long_hits >= int(min_long_confluence_hits)
    confluence_ok = ((int(candidate["confluence_count"]) >= int(confluence_required)) and multi_horizon_ok) or bool(candidate["breaks_structure"])
    matured_enough = bars_since_candidate >= int(force_confirmation_after_bars)

    local_col = f"rolling_high_{int(local_swing_window)}" if kind == "high" else f"rolling_low_{int(local_swing_window)}"
    local_level = row.get(local_col, np.nan)
    local_tol = 0.0 if pd.isna(atr) else float(atr) * 0.10
    local_extreme_ok = False
    if not pd.isna(local_level):
        if kind == "high":
            local_extreme_ok = float(candidate["value"]) >= float(local_level) - local_tol
        else:
            local_extreme_ok = float(candidate["value"]) <= float(local_level) + local_tol

    reaction_ok = max(float(metrics["wick"]), float(metrics["close"])) >= float(threshold) * float(local_swing_min_reaction_multiplier)
    local_obvious_ok = local_extreme_ok and reaction_ok and bars_since_candidate >= max(2, int(min_bars) - 1)

    if not confluence_ok and not matured_enough and not local_obvious_ok:
        return None

    if confluence_ok:
        confirmation_path = "confluence"
    elif matured_enough:
        confirmation_path = "forced_maturity"
    else:
        confirmation_path = "local_obvious"

    return {"kind": kind, "available_on": pd.Timestamp(row["date"]), "swing_date": pd.Timestamp(candidate["date"]), "value": float(candidate["value"]), "candidate_index": int(candidate["index"]), "bars_to_confirmation": int(bars_since_candidate), "threshold": float(threshold), "wick_excursion": float(metrics["wick"]), "close_excursion": float(metrics["close"]), "confirmation_price_mode_used": mode_used, "confirmation_path": confirmation_path, "mode": mode, "structure_label": structure_label(kind, float(candidate["value"]), latest_confirmed.get("value")), "breaks_structure": bool(candidate["breaks_structure"]), "reference_confirmed_value": candidate["reference_confirmed_value"], "confluence_count": int(candidate["confluence_count"]), "confluence_short_hits": short_hits, "confluence_long_hits": long_hits, "multi_horizon_ok": bool(multi_horizon_ok), "confluence_windows": ",".join(map(str, candidate["confluence_windows"]))}



def structure_break_threshold(reference_value, atr_value, atr_multiplier, pct_threshold):
    atr_part = 0.0 if pd.isna(atr_value) else float(atr_value) * float(atr_multiplier)
    pct_part = abs(float(reference_value)) * float(pct_threshold)
    return max(atr_part, pct_part)


def check_opposite_structure_break(active_side, state, row, atr, atr_multiplier, pct_threshold):
    if active_side == "high":
        ref = state["latest_confirmed_low"].get("value")
        if ref is None:
            return False, None
        threshold = structure_break_threshold(ref, atr, atr_multiplier, pct_threshold)
        broken = float(row["low"]) < float(ref) - threshold
        return broken, {"kind": "low", "reference_value": ref, "threshold": threshold}
    if active_side == "low":
        ref = state["latest_confirmed_high"].get("value")
        if ref is None:
            return False, None
        threshold = structure_break_threshold(ref, atr, atr_multiplier, pct_threshold)
        broken = float(row["high"]) > float(ref) + threshold
        return broken, {"kind": "high", "reference_value": ref, "threshold": threshold}
    return False, None


def switch_active_side_due_to_break(active_side, state, row, i, atr, stats, windows, tolerance_atr_multiplier, break_meta):
    new_side = opposite_side(active_side)
    state[f"candidate_{active_side}"] = empty_cand(active_side)
    seeded = build_candidate(new_side, row, i, atr, state[f"latest_confirmed_{new_side}"], "opposite_structure_break_switch", windows, tolerance_atr_multiplier)
    state[f"candidate_{new_side}"] = seeded
    state["active_side"] = new_side
    push_candidate_event(stats, new_side, seeded, row["date"], f"candidate_{new_side}_seeded", "opposite_structure_break_switch")
    if seeded.get("breaks_structure", False):
        push_break_attempt_event(stats, new_side, seeded, row["date"])
    stats.setdefault("active_side_switch_events", []).append({
        "available_on": pd.Timestamp(row["date"]),
        "from_side": active_side,
        "to_side": new_side,
        "reference_value": break_meta["reference_value"],
        "threshold": break_meta["threshold"],
    })
    return state, new_side

def choose_bootstrap_confirmation(high_event, low_event):
    if high_event is None and low_event is None:
        return None
    if high_event is None:
        return low_event
    if low_event is None:
        return high_event
    if high_event["candidate_index"] != low_event["candidate_index"]:
        return high_event if high_event["candidate_index"] < low_event["candidate_index"] else low_event
    high_score = (high_event["confluence_count"], -high_event["bars_to_confirmation"])
    low_score = (low_event["confluence_count"], -low_event["bars_to_confirmation"])
    return high_event if high_score >= low_score else low_event


def opposite_side(kind):
    return "low" if kind == "high" else "high"


def apply_confirmation(state, event, row, i, atr, stats, windows, tolerance_atr_multiplier):
    kind = event["kind"]
    other = opposite_side(kind)
    state[f"latest_confirmed_{kind}"] = {"value": float(event["value"]), "date": pd.Timestamp(event["swing_date"]), "index": int(event["candidate_index"]), "confirmed_on": pd.Timestamp(event["available_on"]), "bars_to_confirmation": int(event["bars_to_confirmation"]), "structure_label": event["structure_label"], "breaks_structure": bool(event["breaks_structure"])}
    state[f"candidate_{kind}"] = empty_cand(kind)
    state["active_side"] = other
    state["last_confirmed_side"] = kind
    stats[f"confirmed_{kind}_events"].append(event)
    stats[f"bars_to_confirm_{kind}"].append(int(event["bars_to_confirmation"]))
    seeded_other = build_candidate(other, row, i, atr, state[f"latest_confirmed_{other}"], "post_confirmation_seed", windows, tolerance_atr_multiplier)
    state[f"candidate_{other}"] = seeded_other
    push_candidate_event(stats, other, seeded_other, row["date"], f"candidate_{other}_seeded", "post_confirmation_seed")
    seeded_break = bool(seeded_other["breaks_structure"])
    if seeded_break:
        push_break_attempt_event(stats, other, seeded_other, row["date"])
    return state, other, seeded_break


## 9. Rolling step-by-step simulation

In [ ]:
def snap(row, state):
    ch = state["candidate_high"]
    cl = state["candidate_low"]
    hh = state["latest_confirmed_high"]
    hl = state["latest_confirmed_low"]
    f = state["last_flags"]
    return {"date": pd.Timestamp(row["date"]), "ticker": row["ticker"], "open": float(row["open"]), "high": float(row["high"]), "low": float(row["low"]), "close": float(row["close"]), "volume": np.nan if pd.isna(row["volume"]) else float(row["volume"]), "active_side_expected": state["active_side"], "last_confirmed_side": state["last_confirmed_side"], "candidate_swing_high_value": ch["value"], "candidate_swing_high_date": ch["date"], "candidate_high_age_bars": ch.get("bars_active", 0), "candidate_high_confluence_count": ch.get("confluence_count", 0), "candidate_high_confluence_short_hits": ch.get("confluence_short_hits", 0), "candidate_high_confluence_long_hits": ch.get("confluence_long_hits", 0), "candidate_high_confluence_windows": ",".join(map(str, ch.get("confluence_windows", []))), "candidate_swing_low_value": cl["value"], "candidate_swing_low_date": cl["date"], "candidate_low_age_bars": cl.get("bars_active", 0), "candidate_low_confluence_count": cl.get("confluence_count", 0), "candidate_low_confluence_short_hits": cl.get("confluence_short_hits", 0), "candidate_low_confluence_long_hits": cl.get("confluence_long_hits", 0), "candidate_low_confluence_windows": ",".join(map(str, cl.get("confluence_windows", []))), "candidate_high_break_attempt_active_flag": bool(ch.get("breaks_structure", False)), "candidate_low_break_attempt_active_flag": bool(cl.get("breaks_structure", False)), "confirmed_swing_high_value": hh["value"], "confirmed_swing_high_date": hh["date"], "confirmed_swing_high_structure_label": hh.get("structure_label"), "confirmed_swing_low_value": hl["value"], "confirmed_swing_low_date": hl["date"], "confirmed_swing_low_structure_label": hl.get("structure_label"), "candidate_high_changed_flag": bool(f.get("candidate_high_changed", False)), "candidate_low_changed_flag": bool(f.get("candidate_low_changed", False)), "candidate_high_new_break_attempt_flag": bool(f.get("candidate_high_new_break_attempt", False)), "candidate_low_new_break_attempt_flag": bool(f.get("candidate_low_new_break_attempt", False)), "confirmed_high_new_flag": bool(f.get("confirmed_high_new", False)), "confirmed_low_new_flag": bool(f.get("confirmed_low_new", False)), "confirmed_high_new_label": f.get("confirmed_high_new_label"), "confirmed_low_new_label": f.get("confirmed_low_new_label"), "confirmed_high_reference_candidate_date": f.get("confirmed_high_reference_candidate_date", pd.NaT), "confirmed_low_reference_candidate_date": f.get("confirmed_low_reference_candidate_date", pd.NaT), "confirmed_high_bars_to_confirmation": f.get("confirmed_high_bars_to_confirmation", np.nan), "confirmed_low_bars_to_confirmation": f.get("confirmed_low_bars_to_confirmation", np.nan)}


def simulate(df, lookback=200, atr_window=14, atr_multiplier=1.5, min_bars_confirmation=3, pct_threshold=0.02, confirmation_mode="atr", level_windows=None, level_confluence_required=2, level_tolerance_atr_multiplier=0.75, max_candidate_bars=25, force_confirmation_after_bars=10, confirmation_price_mode="hybrid", structure_break_switch_atr_multiplier=0.5, structure_break_switch_pct=0.005, local_swing_window=20, local_swing_min_reaction_multiplier=1.0, candidate_replace_min_atr_step=0.15, candidate_replace_min_pct_step=0.001, require_multi_horizon_confluence=False, min_short_confluence_hits=1, min_long_confluence_hits=1):
    if level_windows is None:
        level_windows = [5, 10, 20, 60, 100, 200]
    work = df.copy().reset_index(drop=True)
    work["atr"] = causal_atr(work, atr_window)
    effective_level_windows = sorted(set(list(level_windows) + [int(local_swing_window)]))
    work = build_rolling_levels(work, effective_level_windows)
    state = {"candidate_high": empty_cand("high"), "candidate_low": empty_cand("low"), "latest_confirmed_high": {"value": None, "date": pd.NaT, "index": None, "confirmed_on": pd.NaT, "bars_to_confirmation": np.nan, "structure_label": None, "breaks_structure": False}, "latest_confirmed_low": {"value": None, "date": pd.NaT, "index": None, "confirmed_on": pd.NaT, "bars_to_confirmation": np.nan, "structure_label": None, "breaks_structure": False}, "active_side": "both", "last_confirmed_side": None, "last_flags": {"candidate_high_changed": False, "candidate_low_changed": False, "candidate_high_new_break_attempt": False, "candidate_low_new_break_attempt": False, "confirmed_high_new": False, "confirmed_low_new": False, "confirmed_high_new_label": None, "confirmed_low_new_label": None, "confirmed_high_reference_candidate_date": pd.NaT, "confirmed_low_reference_candidate_date": pd.NaT, "confirmed_high_bars_to_confirmation": np.nan, "confirmed_low_bars_to_confirmation": np.nan}}
    stats = {"candidate_high_events": [], "candidate_low_events": [], "break_attempt_high_events": [], "break_attempt_low_events": [], "confirmed_high_events": [], "confirmed_low_events": [], "candidate_high_replaced_before_confirmation": 0, "candidate_low_replaced_before_confirmation": 0, "bars_to_confirm_high": [], "bars_to_confirm_low": [], "active_side_switch_events": []}
    rows = []
    for i, row in work.iterrows():
        rows.append(snap(row, state))
        atr = float(row["atr"]) if not pd.isna(row["atr"]) else np.nan
        candidate_flags = {"candidate_high_changed": False, "candidate_low_changed": False, "candidate_high_new_break_attempt": False, "candidate_low_new_break_attempt": False}
        confirmed_high_new = False
        confirmed_low_new = False
        confirmed_high_event = None
        confirmed_low_event = None
        if state["active_side"] == "both":
            state, high_meta = update_active_candidate("high", state, row, i, atr, work, lookback, stats, level_windows, level_tolerance_atr_multiplier, max_candidate_bars, candidate_replace_min_atr_step, candidate_replace_min_pct_step)
            state, low_meta = update_active_candidate("low", state, row, i, atr, work, lookback, stats, level_windows, level_tolerance_atr_multiplier, max_candidate_bars, candidate_replace_min_atr_step, candidate_replace_min_pct_step)
            candidate_flags["candidate_high_changed"] = high_meta["changed"]
            candidate_flags["candidate_low_changed"] = low_meta["changed"]
            candidate_flags["candidate_high_new_break_attempt"] = high_meta["new_break"]
            candidate_flags["candidate_low_new_break_attempt"] = low_meta["new_break"]
            high_event = peek_confirmation("high", state, row, i, atr, min_bars_confirmation, confirmation_mode, atr_multiplier, pct_threshold, level_confluence_required, force_confirmation_after_bars, confirmation_price_mode, local_swing_window, local_swing_min_reaction_multiplier, require_multi_horizon_confluence, min_short_confluence_hits, min_long_confluence_hits)
            low_event = peek_confirmation("low", state, row, i, atr, min_bars_confirmation, confirmation_mode, atr_multiplier, pct_threshold, level_confluence_required, force_confirmation_after_bars, confirmation_price_mode, local_swing_window, local_swing_min_reaction_multiplier, require_multi_horizon_confluence, min_short_confluence_hits, min_long_confluence_hits)
            chosen_event = choose_bootstrap_confirmation(high_event, low_event)
            if chosen_event is not None:
                state, seeded_side, seeded_break = apply_confirmation(state, chosen_event, row, i, atr, stats, level_windows, level_tolerance_atr_multiplier)
                candidate_flags[f"candidate_{seeded_side}_changed"] = True
                candidate_flags[f"candidate_{seeded_side}_new_break_attempt"] = seeded_break
                if chosen_event["kind"] == "high":
                    confirmed_high_new = True; confirmed_high_event = chosen_event
                else:
                    confirmed_low_new = True; confirmed_low_event = chosen_event
        else:
            side = state["active_side"]
            broke, break_meta = check_opposite_structure_break(side, state, row, atr, structure_break_switch_atr_multiplier, structure_break_switch_pct)
            if broke:
                state, seeded_side = switch_active_side_due_to_break(side, state, row, i, atr, stats, level_windows, level_tolerance_atr_multiplier, break_meta)
                candidate_flags[f"candidate_{seeded_side}_changed"] = True
                candidate_flags[f"candidate_{seeded_side}_new_break_attempt"] = bool(state[f"candidate_{seeded_side}"].get("breaks_structure", False))
            else:
                state, meta = update_active_candidate(side, state, row, i, atr, work, lookback, stats, level_windows, level_tolerance_atr_multiplier, max_candidate_bars, candidate_replace_min_atr_step, candidate_replace_min_pct_step)
                candidate_flags[f"candidate_{side}_changed"] = meta["changed"]
                candidate_flags[f"candidate_{side}_new_break_attempt"] = meta["new_break"]
                event = peek_confirmation(side, state, row, i, atr, min_bars_confirmation, confirmation_mode, atr_multiplier, pct_threshold, level_confluence_required, force_confirmation_after_bars, confirmation_price_mode, local_swing_window, local_swing_min_reaction_multiplier, require_multi_horizon_confluence, min_short_confluence_hits, min_long_confluence_hits)
                if event is not None:
                    state, seeded_side, seeded_break = apply_confirmation(state, event, row, i, atr, stats, level_windows, level_tolerance_atr_multiplier)
                    candidate_flags[f"candidate_{seeded_side}_changed"] = True
                    candidate_flags[f"candidate_{seeded_side}_new_break_attempt"] = seeded_break
                    if side == "high":
                        confirmed_high_new = True; confirmed_high_event = event
                    else:
                        confirmed_low_new = True; confirmed_low_event = event
        state["last_flags"] = {"candidate_high_changed": candidate_flags["candidate_high_changed"], "candidate_low_changed": candidate_flags["candidate_low_changed"], "candidate_high_new_break_attempt": candidate_flags["candidate_high_new_break_attempt"], "candidate_low_new_break_attempt": candidate_flags["candidate_low_new_break_attempt"], "confirmed_high_new": confirmed_high_new, "confirmed_low_new": confirmed_low_new, "confirmed_high_new_label": None if confirmed_high_event is None else confirmed_high_event["structure_label"], "confirmed_low_new_label": None if confirmed_low_event is None else confirmed_low_event["structure_label"], "confirmed_high_reference_candidate_date": pd.NaT if confirmed_high_event is None else confirmed_high_event["swing_date"], "confirmed_low_reference_candidate_date": pd.NaT if confirmed_low_event is None else confirmed_low_event["swing_date"], "confirmed_high_bars_to_confirmation": np.nan if confirmed_high_event is None else confirmed_high_event["bars_to_confirmation"], "confirmed_low_bars_to_confirmation": np.nan if confirmed_low_event is None else confirmed_low_event["bars_to_confirmation"]}
    return pd.DataFrame(rows), stats


result_df, swing_stats = simulate(price_df, lookback=ROLLING_LOOKBACK, atr_window=ATR_WINDOW, atr_multiplier=ATR_MULTIPLIER, min_bars_confirmation=MIN_BARS_CONFIRMATION, pct_threshold=PCT_THRESHOLD, confirmation_mode=CONFIRMATION_MODE, level_windows=LEVEL_WINDOWS, level_confluence_required=LEVEL_CONFLUENCE_REQUIRED, level_tolerance_atr_multiplier=LEVEL_TOLERANCE_ATR_MULTIPLIER, max_candidate_bars=MAX_CANDIDATE_BARS, force_confirmation_after_bars=FORCE_CONFIRMATION_AFTER_BARS, confirmation_price_mode=CONFIRMATION_PRICE_MODE, structure_break_switch_atr_multiplier=STRUCTURE_BREAK_SWITCH_ATR_MULTIPLIER, structure_break_switch_pct=STRUCTURE_BREAK_SWITCH_PCT, local_swing_window=LOCAL_SWING_WINDOW, local_swing_min_reaction_multiplier=LOCAL_SWING_MIN_REACTION_MULTIPLIER, candidate_replace_min_atr_step=CANDIDATE_REPLACE_MIN_ATR_STEP, candidate_replace_min_pct_step=CANDIDATE_REPLACE_MIN_PCT_STEP, require_multi_horizon_confluence=REQUIRE_MULTI_HORIZON_CONFLUENCE, min_short_confluence_hits=MIN_SHORT_CONFLUENCE_HITS, min_long_confluence_hits=MIN_LONG_CONFLUENCE_HITS)
print('result shape', result_df.shape)
display(result_df.head(10))
display(result_df.tail(10))


## 10. Visualization

In [ ]:
candidate_high_events_df = pd.DataFrame(swing_stats["candidate_high_events"])
candidate_low_events_df = pd.DataFrame(swing_stats["candidate_low_events"])
break_attempt_high_events_df = pd.DataFrame(swing_stats["break_attempt_high_events"])
break_attempt_low_events_df = pd.DataFrame(swing_stats["break_attempt_low_events"])
confirmed_high_events_df = pd.DataFrame(swing_stats["confirmed_high_events"])
confirmed_low_events_df = pd.DataFrame(swing_stats["confirmed_low_events"])


def filt(events_df, cutoff=None):
    if events_df.empty or cutoff is None:
        return events_df.copy()
    return events_df[events_df["available_on"] <= cutoff].copy()


def filter_candidate_events_for_plot(events_df, confluence_required=2, mode="significant", min_gap_days=5, require_multi_horizon_confluence=False, min_short_confluence_hits=1, min_long_confluence_hits=1, min_confluence_for_candidate=3, include_seeded_candidates=False):
    if events_df.empty:
        return events_df.copy()
    mode = str(mode).lower()
    out = events_df.copy()
    if mode == "break_only":
        out = out[out["breaks_structure"].fillna(False)]
    elif mode == "significant":
        seeded_or_reanchored = out["event_type"].astype(str).str.contains("seeded|reanchored", regex=True, na=False)
        structure_break = out["breaks_structure"].fillna(False)
        confluence_floor = max(int(confluence_required), int(min_confluence_for_candidate))
        strong_confluence = out["confluence_count"].fillna(0) >= confluence_floor
        short_hits = out["confluence_short_hits"].fillna(0) if "confluence_short_hits" in out.columns else pd.Series(0, index=out.index)
        long_hits = out["confluence_long_hits"].fillna(0) if "confluence_long_hits" in out.columns else pd.Series(0, index=out.index)
        multi_horizon_ok = pd.Series(True, index=out.index)
        if bool(require_multi_horizon_confluence):
            multi_horizon_ok = (short_hits >= int(min_short_confluence_hits)) & (long_hits >= int(min_long_confluence_hits))
        non_replace = ~out["event_type"].astype(str).str.endswith("_replaced")
        seeded_gate = seeded_or_reanchored if bool(include_seeded_candidates) else pd.Series(False, index=out.index)
        out = out[seeded_gate | structure_break | (strong_confluence & non_replace & multi_horizon_ok)]
    out = out.sort_values(["swing_date", "available_on"]).drop_duplicates(subset=["swing_date", "value"], keep="last")
    if mode == "significant" and not out.empty and int(min_gap_days) > 0:
        keep_idx = []
        last_kept_date = None
        for idx, row in out.iterrows():
            current_date = pd.Timestamp(row["swing_date"])
            if last_kept_date is None:
                keep_idx.append(idx)
                last_kept_date = current_date
                continue
            if (current_date - last_kept_date).days >= int(min_gap_days):
                keep_idx.append(idx)
                last_kept_date = current_date
        out = out.loc[keep_idx]
    return out


def filter_break_attempt_events_for_plot(events_df, mode="significant", min_confluence=3, min_gap_days=8, require_multi_horizon_confluence=False, min_short_confluence_hits=1, min_long_confluence_hits=1):
    if events_df.empty:
        return events_df.copy()
    mode = str(mode).lower()
    if mode == "off":
        return events_df.iloc[0:0].copy()
    out = events_df.copy()
    if mode == "significant":
        strong_confluence = out["confluence_count"].fillna(0) >= int(min_confluence)
        short_hits = out["confluence_short_hits"].fillna(0) if "confluence_short_hits" in out.columns else pd.Series(0, index=out.index)
        long_hits = out["confluence_long_hits"].fillna(0) if "confluence_long_hits" in out.columns else pd.Series(0, index=out.index)
        multi_horizon_ok = pd.Series(True, index=out.index)
        if bool(require_multi_horizon_confluence):
            multi_horizon_ok = (short_hits >= int(min_short_confluence_hits)) & (long_hits >= int(min_long_confluence_hits))
        out = out[strong_confluence & multi_horizon_ok]
    out = out.sort_values(["swing_date", "available_on"]).drop_duplicates(subset=["swing_date", "value"], keep="last")
    if mode == "significant" and int(min_gap_days) > 0 and not out.empty:
        keep_idx = []
        last_kept_date = None
        for idx, row in out.iterrows():
            current_date = pd.Timestamp(row["swing_date"])
            if last_kept_date is None or (current_date - last_kept_date).days >= int(min_gap_days):
                keep_idx.append(idx)
                last_kept_date = current_date
        out = out.loc[keep_idx]
    return out


def parse_confluence_windows(value):
    if pd.isna(value):
        return []
    text = str(value).strip()
    if text == "":
        return []
    out = []
    for token in text.split(","):
        token = token.strip()
        if token.lstrip("-").isdigit():
            out.append(int(token))
    return out


def classify_confirmed_tier(events_df, structural_min_total=3, structural_min_long_hits=1, global_min_total=4, global_min_long_hits=2, global_require_200=False):
    if events_df.empty:
        return events_df.copy()
    out = events_df.copy()
    if "confluence_count" not in out.columns:
        out["confluence_count"] = 0
    if "confluence_long_hits" not in out.columns:
        out["confluence_long_hits"] = 0
    if "confluence_windows" not in out.columns:
        out["confluence_windows"] = ""
    windows = out["confluence_windows"].apply(parse_confluence_windows)
    has_200 = windows.apply(lambda ws: 200 in ws)
    global_gate = (out["confluence_count"].fillna(0) >= int(global_min_total)) & (out["confluence_long_hits"].fillna(0) >= int(global_min_long_hits))
    if bool(global_require_200):
        global_gate = global_gate & has_200
    structural_gate = (out["confluence_count"].fillna(0) >= int(structural_min_total)) & (out["confluence_long_hits"].fillna(0) >= int(structural_min_long_hits))
    out["swing_tier"] = np.where(global_gate, "global_extrema", np.where(structural_gate, "structural_extrema", "local_extrema"))
    return out


def relabel_major_structure(events_df, kind, structural_min_total=3, structural_min_long_hits=1, global_min_total=4, global_min_long_hits=2, global_require_200=False):
    if events_df.empty:
        return events_df.copy()
    out = classify_confirmed_tier(events_df, structural_min_total, structural_min_long_hits, global_min_total, global_min_long_hits, global_require_200)
    if "structure_label" not in out.columns:
        out["structure_label"] = ("INITIAL_HIGH" if kind == "high" else "INITIAL_LOW")
    out = out.sort_values(["swing_date", "available_on"]).copy()
    out["structure_label_sequential"] = out["structure_label"].astype(str)
    out["structure_label_major"] = out["structure_label_sequential"].copy()
    out["structure_reference_value"] = np.nan
    out["structure_reference_date"] = pd.NaT
    major_mask = out["swing_tier"].isin(["structural_extrema", "global_extrema"])
    major = out[major_mask].sort_values(["swing_date", "available_on"])
    anchor_value = None
    anchor_date = pd.NaT
    for idx, row in major.iterrows():
        value = float(row.get("value", np.nan))
        if not np.isfinite(value):
            continue
        out.at[idx, "structure_reference_value"] = anchor_value if anchor_value is not None else np.nan
        out.at[idx, "structure_reference_date"] = anchor_date if anchor_value is not None else pd.NaT
        if anchor_value is None:
            label = "INITIAL_HIGH" if kind == "high" else "INITIAL_LOW"
            anchor_value = value
            anchor_date = pd.Timestamp(row["swing_date"])
        else:
            if kind == "high":
                label = "HH" if value >= anchor_value else "LH"
                if value > anchor_value:
                    anchor_value = value
                    anchor_date = pd.Timestamp(row["swing_date"])
            else:
                label = "LL" if value <= anchor_value else "HL"
                if value < anchor_value:
                    anchor_value = value
                    anchor_date = pd.Timestamp(row["swing_date"])
        out.at[idx, "structure_label_major"] = label
    out["structure_label"] = np.where(major_mask, out["structure_label_major"], out["structure_label_sequential"])
    return out.sort_values(["swing_date", "available_on"]).reset_index(drop=True)


def _micro_tier_filter(df, tier_mode="all"):
    mode = str(tier_mode).lower()
    if mode == "structural_plus":
        return df[df["swing_tier"].isin(["structural_extrema", "global_extrema"])].copy()
    if mode == "local_plus":
        return df[df["swing_tier"].isin(["local_extrema", "structural_extrema"])].copy()
    return df.copy()


def _micro_tier_bonus(tier):
    t = str(tier)
    if t == "global_extrema":
        return 1.25
    if t == "structural_extrema":
        return 0.75
    return 0.25


def assign_micro_structure_labels(events_df, kind, lookback_events=7, min_confluence=2, tier_mode="all"):
    if events_df.empty:
        return events_df.copy()
    out = events_df.sort_values(["swing_date", "available_on"]).reset_index(drop=True).copy()
    if "structure_label_sequential" not in out.columns:
        out["structure_label_sequential"] = out.get("structure_label", pd.Series("", index=out.index)).astype(str)
    out["structure_label_micro"] = out["structure_label_sequential"].astype(str)
    out["structure_label_micro_source"] = "sequential"
    out["micro_reference_value"] = np.nan
    out["micro_reference_count"] = 0
    out["micro_reference_tiers"] = ""
    n = max(2, int(lookback_events))
    min_conf = max(0, int(min_confluence))
    for i in range(len(out)):
        row = out.iloc[i]
        value = float(row.get("value", np.nan))
        if not np.isfinite(value):
            continue
        hist = out.iloc[:i].copy()
        if hist.empty:
            out.at[i, "structure_label_micro"] = ("INITIAL_HIGH" if kind == "high" else "INITIAL_LOW")
            out.at[i, "structure_label_micro_source"] = "initial"
            continue
        hist = hist[hist.get("confluence_count", pd.Series(0, index=hist.index)).fillna(0) >= min_conf]
        hist = _micro_tier_filter(hist, tier_mode)
        hist = hist.tail(n)
        if hist.empty:
            out.at[i, "structure_label_micro"] = str(row.get("structure_label_sequential", out.at[i, "structure_label_micro"]))
            out.at[i, "structure_label_micro_source"] = "sequential_fallback"
            continue
        weights = 1.0 + hist.get("confluence_count", pd.Series(0, index=hist.index)).fillna(0).astype(float)
        weights = weights + hist.get("swing_tier", pd.Series("", index=hist.index)).astype(str).apply(_micro_tier_bonus)
        ref = float(np.average(hist["value"].astype(float), weights=weights.astype(float)))
        tiers = sorted(set(hist.get("swing_tier", pd.Series("", index=hist.index)).astype(str)))
        out.at[i, "micro_reference_value"] = ref
        out.at[i, "micro_reference_count"] = int(len(hist))
        out.at[i, "micro_reference_tiers"] = ",".join(tiers)
        if kind == "high":
            out.at[i, "structure_label_micro"] = "HH" if value >= ref else "LH"
        else:
            out.at[i, "structure_label_micro"] = "LL" if value <= ref else "HL"
        out.at[i, "structure_label_micro_source"] = "weighted_confluence_ref"
    return out


confirmed_high_events_df = relabel_major_structure(confirmed_high_events_df, "high", PLOT_STRUCTURAL_MIN_TOTAL_CONFLUENCE, PLOT_STRUCTURAL_MIN_LONG_HITS, PLOT_GLOBAL_MIN_TOTAL_CONFLUENCE, PLOT_GLOBAL_MIN_LONG_HITS, PLOT_GLOBAL_REQUIRE_200_WINDOW)
confirmed_low_events_df = relabel_major_structure(confirmed_low_events_df, "low", PLOT_STRUCTURAL_MIN_TOTAL_CONFLUENCE, PLOT_STRUCTURAL_MIN_LONG_HITS, PLOT_GLOBAL_MIN_TOTAL_CONFLUENCE, PLOT_GLOBAL_MIN_LONG_HITS, PLOT_GLOBAL_REQUIRE_200_WINDOW)
confirmed_high_events_df = assign_micro_structure_labels(confirmed_high_events_df, "high", MICRO_STRUCTURE_EVENT_LOOKBACK, MICRO_STRUCTURE_MIN_CONFLUENCE, MICRO_STRUCTURE_TIER_MODE)
confirmed_low_events_df = assign_micro_structure_labels(confirmed_low_events_df, "low", MICRO_STRUCTURE_EVENT_LOOKBACK, MICRO_STRUCTURE_MIN_CONFLUENCE, MICRO_STRUCTURE_TIER_MODE)


def filter_confirmed_events_for_plot(events_df, tier_filter="all", structural_min_total=3, structural_min_long_hits=1, global_min_total=4, global_min_long_hits=2, global_require_200=False, min_gap_days_local=0, min_gap_days_structural=0, min_gap_days_global=0):
    if events_df.empty:
        return events_df.copy()
    out = classify_confirmed_tier(events_df, structural_min_total, structural_min_long_hits, global_min_total, global_min_long_hits, global_require_200)
    mode = str(tier_filter).lower()
    if mode == "global_only":
        out = out[out["swing_tier"] == "global_extrema"]
    elif mode == "structural_plus":
        out = out[out["swing_tier"].isin(["structural_extrema", "global_extrema"])]
    out = out.sort_values(["swing_date", "available_on"]).drop_duplicates(subset=["swing_date", "value"], keep="last")
    gap_map = {
        "local_extrema": int(min_gap_days_local),
        "structural_extrema": int(min_gap_days_structural),
        "global_extrema": int(min_gap_days_global),
    }
    keep = []
    for tier, grp in out.groupby("swing_tier", dropna=False):
        g = grp.sort_values(["swing_date", "available_on"])
        min_gap = max(0, gap_map.get(tier, 0))
        last_kept_date = None
        for idx, row in g.iterrows():
            current_date = pd.Timestamp(row["swing_date"])
            if last_kept_date is None or (current_date - last_kept_date).days >= min_gap:
                keep.append(idx)
                last_kept_date = current_date
    if keep:
        out = out.loc[sorted(set(keep))]
    else:
        out = out.iloc[0:0].copy()
    return out.sort_values(["swing_date", "available_on"])


def add_confirmed_traces(fig, events_df, kind, style_mode="tiered"):
    if events_df.empty:
        return
    kind_title = "High" if kind == "high" else "Low"
    base_symbol = "x" if kind == "high" else "circle"
    if style_mode == "basic":
        base_color = "red" if kind == "high" else "green"
        fig.add_trace(go.Scatter(x=events_df["swing_date"], y=events_df["value"], mode="markers", name=f"Confirmed {kind_title}", marker=dict(color=base_color, symbol=base_symbol, size=11), text=events_df.get("structure_label", pd.Series("", index=events_df.index)), hovertemplate=f"Confirmed {kind_title}<br>Date=%{{x}}<br>Value=%{{y:.2f}}<br>Label=%{{text}}<extra></extra>"))
        return
    if kind == "high":
        styles = [
            ("global_extrema", "Confirmed High (Global)", dict(color="#9b2226", symbol="x", size=14, line=dict(width=2, color="#5f0f40"))),
            ("structural_extrema", "Confirmed High (Structural)", dict(color="#d62828", symbol="x", size=12, line=dict(width=1, color="#9d0208"))),
            ("local_extrema", "Confirmed High (Local)", dict(color="#f4a261", symbol="x-open", size=8, opacity=0.65, line=dict(width=1, color="#bc6c25"))),
        ]
    else:
        styles = [
            ("global_extrema", "Confirmed Low (Global)", dict(color="#1b4332", symbol="circle", size=13, line=dict(width=2, color="#081c15"))),
            ("structural_extrema", "Confirmed Low (Structural)", dict(color="#2d6a4f", symbol="circle", size=11, line=dict(width=1, color="#1b4332"))),
            ("local_extrema", "Confirmed Low (Local)", dict(color="#95d5b2", symbol="circle-open", size=8, opacity=0.65, line=dict(width=1, color="#40916c"))),
        ]
    for tier, trace_name, marker_style in styles:
        part = events_df[events_df["swing_tier"] == tier]
        if part.empty:
            continue
        labels = part.get("structure_label", pd.Series("", index=part.index)).astype(str)
        seq_labels = part.get("structure_label_sequential", labels).astype(str)
        major_labels = part.get("structure_label_major", labels).astype(str)
        micro_labels = part.get("structure_label_micro", labels).astype(str)
        micro_source = part.get("structure_label_micro_source", pd.Series("", index=part.index)).astype(str)
        micro_ref_n = part.get("micro_reference_count", pd.Series(0, index=part.index)).fillna(0).astype(int).astype(str)
        ccount = part.get("confluence_count", pd.Series(0, index=part.index)).fillna(0).astype(int).astype(str)
        tier_text = part.get("swing_tier", pd.Series("", index=part.index)).astype(str)
        hover_text = labels + " | seq=" + seq_labels + " | major=" + major_labels + " | micro=" + micro_labels + " (" + micro_source + ", n=" + micro_ref_n + ") | Confluence=" + ccount + " | " + tier_text
        fig.add_trace(go.Scatter(x=part["swing_date"], y=part["value"], mode="markers", name=trace_name, marker=marker_style, text=hover_text, hovertemplate=f"{trace_name}<br>Date=%{{x}}<br>Value=%{{y:.2f}}<br>%{{text}}<extra></extra>"))


def structure_figure(data, ch, cl, hh, hl, bh, bl, title, confirmed_style_mode="tiered"):
    fig = go.Figure()
    fig.add_trace(go.Candlestick(x=data["date"], open=data["open"], high=data["high"], low=data["low"], close=data["close"], name="Price"))
    if not ch.empty:
        fig.add_trace(go.Scatter(x=ch["swing_date"], y=ch["value"], mode="markers", name="Candidate High", marker=dict(color="orange", symbol="triangle-up", size=PLOT_CANDIDATE_MARKER_SIZE), opacity=PLOT_CANDIDATE_MARKER_OPACITY, text=ch["confluence_windows"], hovertemplate="Candidate High<br>Date=%{x}<br>Value=%{y:.2f}<br>Confluence=%{text}<extra></extra>"))
    if not cl.empty:
        fig.add_trace(go.Scatter(x=cl["swing_date"], y=cl["value"], mode="markers", name="Candidate Low", marker=dict(color="cyan", symbol="triangle-down", size=PLOT_CANDIDATE_MARKER_SIZE), opacity=PLOT_CANDIDATE_MARKER_OPACITY, text=cl["confluence_windows"], hovertemplate="Candidate Low<br>Date=%{x}<br>Value=%{y:.2f}<br>Confluence=%{text}<extra></extra>"))
    if not bh.empty:
        fig.add_trace(go.Scatter(x=bh["swing_date"], y=bh["value"], mode="markers", name="Fresh Break-Up Attempt", marker=dict(color="darkorange", symbol="star", size=13), text=bh["confluence_windows"], hovertemplate="Break-Up Attempt<br>Date=%{x}<br>Value=%{y:.2f}<br>Confluence=%{text}<extra></extra>"))
    if not bl.empty:
        fig.add_trace(go.Scatter(x=bl["swing_date"], y=bl["value"], mode="markers", name="Fresh Break-Down Attempt", marker=dict(color="deepskyblue", symbol="star", size=13), text=bl["confluence_windows"], hovertemplate="Break-Down Attempt<br>Date=%{x}<br>Value=%{y:.2f}<br>Confluence=%{text}<extra></extra>"))
    add_confirmed_traces(fig, hh, "high", confirmed_style_mode)
    add_confirmed_traces(fig, hl, "low", confirmed_style_mode)
    fig.update_layout(title=title, template="plotly_white", height=720, xaxis_title="Date", yaxis_title="Price", xaxis_rangeslider_visible=False, legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0))
    return fig


def plot_last_n_bars(last_n=PLOT_LAST_N_BARS):
    data = result_df.tail(last_n).copy(); start = data["date"].min()
    ch_raw = candidate_high_events_df[candidate_high_events_df["swing_date"] >= start] if not candidate_high_events_df.empty else candidate_high_events_df
    cl_raw = candidate_low_events_df[candidate_low_events_df["swing_date"] >= start] if not candidate_low_events_df.empty else candidate_low_events_df
    ch = filter_candidate_events_for_plot(ch_raw, LEVEL_CONFLUENCE_REQUIRED, PLOT_CANDIDATE_MODE, PLOT_SIGNIFICANT_MIN_GAP_DAYS, REQUIRE_MULTI_HORIZON_CONFLUENCE, MIN_SHORT_CONFLUENCE_HITS, MIN_LONG_CONFLUENCE_HITS, PLOT_MIN_CONFLUENCE_FOR_CANDIDATE, PLOT_INCLUDE_SEEDED_CANDIDATES)
    cl = filter_candidate_events_for_plot(cl_raw, LEVEL_CONFLUENCE_REQUIRED, PLOT_CANDIDATE_MODE, PLOT_SIGNIFICANT_MIN_GAP_DAYS, REQUIRE_MULTI_HORIZON_CONFLUENCE, MIN_SHORT_CONFLUENCE_HITS, MIN_LONG_CONFLUENCE_HITS, PLOT_MIN_CONFLUENCE_FOR_CANDIDATE, PLOT_INCLUDE_SEEDED_CANDIDATES)
    hh_raw = confirmed_high_events_df[confirmed_high_events_df["swing_date"] >= start] if not confirmed_high_events_df.empty else confirmed_high_events_df
    hl_raw = confirmed_low_events_df[confirmed_low_events_df["swing_date"] >= start] if not confirmed_low_events_df.empty else confirmed_low_events_df
    hh = filter_confirmed_events_for_plot(hh_raw, PLOT_CONFIRMED_TIER_FILTER, PLOT_STRUCTURAL_MIN_TOTAL_CONFLUENCE, PLOT_STRUCTURAL_MIN_LONG_HITS, PLOT_GLOBAL_MIN_TOTAL_CONFLUENCE, PLOT_GLOBAL_MIN_LONG_HITS, PLOT_GLOBAL_REQUIRE_200_WINDOW, PLOT_CONFIRMED_MIN_GAP_DAYS_LOCAL, PLOT_CONFIRMED_MIN_GAP_DAYS_STRUCTURAL, PLOT_CONFIRMED_MIN_GAP_DAYS_GLOBAL)
    hl = filter_confirmed_events_for_plot(hl_raw, PLOT_CONFIRMED_TIER_FILTER, PLOT_STRUCTURAL_MIN_TOTAL_CONFLUENCE, PLOT_STRUCTURAL_MIN_LONG_HITS, PLOT_GLOBAL_MIN_TOTAL_CONFLUENCE, PLOT_GLOBAL_MIN_LONG_HITS, PLOT_GLOBAL_REQUIRE_200_WINDOW, PLOT_CONFIRMED_MIN_GAP_DAYS_LOCAL, PLOT_CONFIRMED_MIN_GAP_DAYS_STRUCTURAL, PLOT_CONFIRMED_MIN_GAP_DAYS_GLOBAL)
    bh_raw = break_attempt_high_events_df[break_attempt_high_events_df["swing_date"] >= start] if not break_attempt_high_events_df.empty else break_attempt_high_events_df
    bl_raw = break_attempt_low_events_df[break_attempt_low_events_df["swing_date"] >= start] if not break_attempt_low_events_df.empty else break_attempt_low_events_df
    bh = filter_break_attempt_events_for_plot(bh_raw, PLOT_BREAK_ATTEMPT_MODE, PLOT_BREAK_MIN_CONFLUENCE, PLOT_BREAK_MIN_GAP_DAYS, REQUIRE_MULTI_HORIZON_CONFLUENCE, MIN_SHORT_CONFLUENCE_HITS, MIN_LONG_CONFLUENCE_HITS)
    bl = filter_break_attempt_events_for_plot(bl_raw, PLOT_BREAK_ATTEMPT_MODE, PLOT_BREAK_MIN_CONFLUENCE, PLOT_BREAK_MIN_GAP_DAYS, REQUIRE_MULTI_HORIZON_CONFLUENCE, MIN_SHORT_CONFLUENCE_HITS, MIN_LONG_CONFLUENCE_HITS)
    return structure_figure(data, ch, cl, hh, hl, bh, bl, f"Static structure view - last {last_n} bars", PLOT_CONFIRMED_STYLE_MODE)


def plot_until_index(i, trailing_bars=PLOT_LAST_N_BARS):
    if i < 0 or i >= len(result_df): raise IndexError(f"i must be between 0 and {len(result_df)-1}")
    cutoff = pd.Timestamp(result_df.iloc[i]["date"]); start = max(0, i - trailing_bars + 1); data = result_df.iloc[start : i + 1].copy()
    ch = filter_candidate_events_for_plot(filt(candidate_high_events_df, cutoff), LEVEL_CONFLUENCE_REQUIRED, PLOT_CANDIDATE_MODE, PLOT_SIGNIFICANT_MIN_GAP_DAYS, REQUIRE_MULTI_HORIZON_CONFLUENCE, MIN_SHORT_CONFLUENCE_HITS, MIN_LONG_CONFLUENCE_HITS, PLOT_MIN_CONFLUENCE_FOR_CANDIDATE, PLOT_INCLUDE_SEEDED_CANDIDATES)
    cl = filter_candidate_events_for_plot(filt(candidate_low_events_df, cutoff), LEVEL_CONFLUENCE_REQUIRED, PLOT_CANDIDATE_MODE, PLOT_SIGNIFICANT_MIN_GAP_DAYS, REQUIRE_MULTI_HORIZON_CONFLUENCE, MIN_SHORT_CONFLUENCE_HITS, MIN_LONG_CONFLUENCE_HITS, PLOT_MIN_CONFLUENCE_FOR_CANDIDATE, PLOT_INCLUDE_SEEDED_CANDIDATES)
    bh = filter_break_attempt_events_for_plot(filt(break_attempt_high_events_df, cutoff), PLOT_BREAK_ATTEMPT_MODE, PLOT_BREAK_MIN_CONFLUENCE, PLOT_BREAK_MIN_GAP_DAYS, REQUIRE_MULTI_HORIZON_CONFLUENCE, MIN_SHORT_CONFLUENCE_HITS, MIN_LONG_CONFLUENCE_HITS)
    bl = filter_break_attempt_events_for_plot(filt(break_attempt_low_events_df, cutoff), PLOT_BREAK_ATTEMPT_MODE, PLOT_BREAK_MIN_CONFLUENCE, PLOT_BREAK_MIN_GAP_DAYS, REQUIRE_MULTI_HORIZON_CONFLUENCE, MIN_SHORT_CONFLUENCE_HITS, MIN_LONG_CONFLUENCE_HITS)
    hh = filter_confirmed_events_for_plot(filt(confirmed_high_events_df, cutoff), PLOT_CONFIRMED_TIER_FILTER, PLOT_STRUCTURAL_MIN_TOTAL_CONFLUENCE, PLOT_STRUCTURAL_MIN_LONG_HITS, PLOT_GLOBAL_MIN_TOTAL_CONFLUENCE, PLOT_GLOBAL_MIN_LONG_HITS, PLOT_GLOBAL_REQUIRE_200_WINDOW, PLOT_CONFIRMED_MIN_GAP_DAYS_LOCAL, PLOT_CONFIRMED_MIN_GAP_DAYS_STRUCTURAL, PLOT_CONFIRMED_MIN_GAP_DAYS_GLOBAL)
    hl = filter_confirmed_events_for_plot(filt(confirmed_low_events_df, cutoff), PLOT_CONFIRMED_TIER_FILTER, PLOT_STRUCTURAL_MIN_TOTAL_CONFLUENCE, PLOT_STRUCTURAL_MIN_LONG_HITS, PLOT_GLOBAL_MIN_TOTAL_CONFLUENCE, PLOT_GLOBAL_MIN_LONG_HITS, PLOT_GLOBAL_REQUIRE_200_WINDOW, PLOT_CONFIRMED_MIN_GAP_DAYS_LOCAL, PLOT_CONFIRMED_MIN_GAP_DAYS_STRUCTURAL, PLOT_CONFIRMED_MIN_GAP_DAYS_GLOBAL)
    return structure_figure(data, ch, cl, hh, hl, bh, bl, f"Step-by-step causal view up to index {i} ({cutoff.date()})", PLOT_CONFIRMED_STYLE_MODE)


# plots are executed in separate cells below


## 11. Validation / diagnostics

In [ ]:
high_label_counts = confirmed_high_events_df["structure_label"].value_counts(dropna=False) if not confirmed_high_events_df.empty else pd.Series(dtype=int)
low_label_counts = confirmed_low_events_df["structure_label"].value_counts(dropna=False) if not confirmed_low_events_df.empty else pd.Series(dtype=int)
high_micro_label_counts = confirmed_high_events_df.get("structure_label_micro", pd.Series(dtype=str)).value_counts(dropna=False) if not confirmed_high_events_df.empty else pd.Series(dtype=int)
low_micro_label_counts = confirmed_low_events_df.get("structure_label_micro", pd.Series(dtype=str)).value_counts(dropna=False) if not confirmed_low_events_df.empty else pd.Series(dtype=int)
confirmation_path_counts = pd.concat([confirmed_high_events_df.get("confirmation_path", pd.Series(dtype=str)), confirmed_low_events_df.get("confirmation_path", pd.Series(dtype=str))]).value_counts(dropna=False)
multi_horizon_confirmed = int(pd.concat([confirmed_high_events_df.get("multi_horizon_ok", pd.Series(dtype=bool)), confirmed_low_events_df.get("multi_horizon_ok", pd.Series(dtype=bool))]).fillna(False).sum())
confirmed_tier_df = classify_confirmed_tier(pd.concat([confirmed_high_events_df, confirmed_low_events_df], ignore_index=True) if (not confirmed_high_events_df.empty or not confirmed_low_events_df.empty) else pd.DataFrame(), PLOT_STRUCTURAL_MIN_TOTAL_CONFLUENCE, PLOT_STRUCTURAL_MIN_LONG_HITS, PLOT_GLOBAL_MIN_TOTAL_CONFLUENCE, PLOT_GLOBAL_MIN_LONG_HITS, PLOT_GLOBAL_REQUIRE_200_WINDOW)
tier_counts = confirmed_tier_df["swing_tier"].value_counts(dropna=False) if not confirmed_tier_df.empty else pd.Series(dtype=int)
summary_metrics = pd.DataFrame([
    {"metric": "total_candidate_high_changes", "value": int(result_df["candidate_high_changed_flag"].sum())},
    {"metric": "total_candidate_low_changes", "value": int(result_df["candidate_low_changed_flag"].sum())},
    {"metric": "fresh_break_attempt_highs", "value": len(break_attempt_high_events_df)},
    {"metric": "fresh_break_attempt_lows", "value": len(break_attempt_low_events_df)},
    {"metric": "active_break_rows_high", "value": int(result_df["candidate_high_break_attempt_active_flag"].sum())},
    {"metric": "active_break_rows_low", "value": int(result_df["candidate_low_break_attempt_active_flag"].sum())},
    {"metric": "total_confirmed_highs", "value": int(result_df["confirmed_high_new_flag"].sum())},
    {"metric": "total_confirmed_lows", "value": int(result_df["confirmed_low_new_flag"].sum())},
    {"metric": "avg_bars_until_high_confirmation", "value": float(np.mean(swing_stats["bars_to_confirm_high"])) if swing_stats["bars_to_confirm_high"] else np.nan},
    {"metric": "avg_bars_until_low_confirmation", "value": float(np.mean(swing_stats["bars_to_confirm_low"])) if swing_stats["bars_to_confirm_low"] else np.nan},
    {"metric": "candidate_high_replaced_before_confirmation", "value": int(swing_stats["candidate_high_replaced_before_confirmation"])},
    {"metric": "candidate_low_replaced_before_confirmation", "value": int(swing_stats["candidate_low_replaced_before_confirmation"])},
    {"metric": "avg_candidate_high_confluence", "value": float(result_df["candidate_high_confluence_count"].replace(0, np.nan).mean())},
    {"metric": "avg_candidate_low_confluence", "value": float(result_df["candidate_low_confluence_count"].replace(0, np.nan).mean())},
    {"metric": "confirmed_HH_count", "value": int(high_label_counts.get("HH", 0))},
    {"metric": "confirmed_LH_count", "value": int(high_label_counts.get("LH", 0))},
    {"metric": "confirmed_HL_count", "value": int(low_label_counts.get("HL", 0))},
    {"metric": "confirmed_LL_count", "value": int(low_label_counts.get("LL", 0))},
    {"metric": "confirmed_micro_HH_count", "value": int(high_micro_label_counts.get("HH", 0))},
    {"metric": "confirmed_micro_LH_count", "value": int(high_micro_label_counts.get("LH", 0))},
    {"metric": "confirmed_micro_HL_count", "value": int(low_micro_label_counts.get("HL", 0))},
    {"metric": "confirmed_micro_LL_count", "value": int(low_micro_label_counts.get("LL", 0))},
    {"metric": "confirmation_path_confluence", "value": int(confirmation_path_counts.get("confluence", 0))},
    {"metric": "confirmation_path_forced_maturity", "value": int(confirmation_path_counts.get("forced_maturity", 0))},
    {"metric": "confirmation_path_local_obvious", "value": int(confirmation_path_counts.get("local_obvious", 0))},
    {"metric": "confirmed_multi_horizon_ok", "value": multi_horizon_confirmed},
    {"metric": "confirmed_global_extrema_count", "value": int(tier_counts.get("global_extrema", 0))},
    {"metric": "confirmed_structural_extrema_count", "value": int(tier_counts.get("structural_extrema", 0))},
    {"metric": "confirmed_local_extrema_count", "value": int(tier_counts.get("local_extrema", 0))},
])
display(summary_metrics)
candidate_change_rows = result_df[result_df["candidate_high_changed_flag"] | result_df["candidate_low_changed_flag"]].copy()
confirmed_change_rows = result_df[result_df["confirmed_high_new_flag"] | result_df["confirmed_low_new_flag"]].copy()
fresh_break_rows = result_df[result_df["candidate_high_new_break_attempt_flag"] | result_df["candidate_low_new_break_attempt_flag"]].copy()
print("Candidate state change rows:"); display(candidate_change_rows.head(20))
print("Confirmed state change rows:"); display(confirmed_change_rows.head(20))
print("Fresh break attempt rows:"); display(fresh_break_rows.head(20))
print("Micro structure settings:", {"lookback_events": MICRO_STRUCTURE_EVENT_LOOKBACK, "min_confluence": MICRO_STRUCTURE_MIN_CONFLUENCE, "tier_mode": MICRO_STRUCTURE_TIER_MODE})
if not confirmed_high_events_df.empty:
    high_tier = classify_confirmed_tier(confirmed_high_events_df, PLOT_STRUCTURAL_MIN_TOTAL_CONFLUENCE, PLOT_STRUCTURAL_MIN_LONG_HITS, PLOT_GLOBAL_MIN_TOTAL_CONFLUENCE, PLOT_GLOBAL_MIN_LONG_HITS, PLOT_GLOBAL_REQUIRE_200_WINDOW)
    print("Confirmed high events:"); display(high_tier[["available_on", "swing_date", "value", "structure_label", "structure_label_major", "structure_label_micro", "structure_label_micro_source", "swing_tier", "bars_to_confirmation", "confluence_short_hits", "confluence_long_hits", "multi_horizon_ok", "confluence_windows", "confirmation_price_mode_used"]].head(20))
if not confirmed_low_events_df.empty:
    low_tier = classify_confirmed_tier(confirmed_low_events_df, PLOT_STRUCTURAL_MIN_TOTAL_CONFLUENCE, PLOT_STRUCTURAL_MIN_LONG_HITS, PLOT_GLOBAL_MIN_TOTAL_CONFLUENCE, PLOT_GLOBAL_MIN_LONG_HITS, PLOT_GLOBAL_REQUIRE_200_WINDOW)
    print("Confirmed low events:"); display(low_tier[["available_on", "swing_date", "value", "structure_label", "structure_label_major", "structure_label_micro", "structure_label_micro_source", "swing_tier", "bars_to_confirmation", "confluence_short_hits", "confluence_long_hits", "multi_horizon_ok", "confluence_windows", "confirmation_price_mode_used"]].head(20))
if not break_attempt_high_events_df.empty:
    print("Break-up attempts:"); display(break_attempt_high_events_df.head(20))
if not break_attempt_low_events_df.empty:
    print("Break-down attempts:"); display(break_attempt_low_events_df.head(20))
def show_transition_slice(df, anchor_date, bars_before=5, bars_after=5):
    idxs = df.index[df["date"] == pd.Timestamp(anchor_date)].tolist()
    if not idxs: raise ValueError(f"Anchor date not found: {anchor_date}")
    idx = idxs[0]; start = max(0, idx - bars_before); end = min(len(df), idx + bars_after + 1); return df.iloc[start:end].copy()
for anchor in confirmed_change_rows["date"].dropna().head(3).tolist():
    print(f"\nTransition slice around {pd.Timestamp(anchor).date()}:")
    display(show_transition_slice(result_df, anchor, 5, 5))

def period_structure_metrics(start_date, end_date):
    start_date = pd.Timestamp(start_date)
    end_date = pd.Timestamp(end_date)
    period_df = result_df[(result_df["date"] >= start_date) & (result_df["date"] <= end_date)].copy()
    bars = len(period_df)
    if bars == 0:
        return None
    ch = filter_candidate_events_for_plot(
        candidate_high_events_df[(candidate_high_events_df["swing_date"] >= start_date) & (candidate_high_events_df["swing_date"] <= end_date)],
        LEVEL_CONFLUENCE_REQUIRED,
        PLOT_CANDIDATE_MODE,
        PLOT_SIGNIFICANT_MIN_GAP_DAYS,
        REQUIRE_MULTI_HORIZON_CONFLUENCE,
        MIN_SHORT_CONFLUENCE_HITS,
        MIN_LONG_CONFLUENCE_HITS,
        PLOT_MIN_CONFLUENCE_FOR_CANDIDATE,
        PLOT_INCLUDE_SEEDED_CANDIDATES,
    )
    cl = filter_candidate_events_for_plot(
        candidate_low_events_df[(candidate_low_events_df["swing_date"] >= start_date) & (candidate_low_events_df["swing_date"] <= end_date)],
        LEVEL_CONFLUENCE_REQUIRED,
        PLOT_CANDIDATE_MODE,
        PLOT_SIGNIFICANT_MIN_GAP_DAYS,
        REQUIRE_MULTI_HORIZON_CONFLUENCE,
        MIN_SHORT_CONFLUENCE_HITS,
        MIN_LONG_CONFLUENCE_HITS,
        PLOT_MIN_CONFLUENCE_FOR_CANDIDATE,
        PLOT_INCLUDE_SEEDED_CANDIDATES,
    )
    bh = filter_break_attempt_events_for_plot(
        break_attempt_high_events_df[(break_attempt_high_events_df["swing_date"] >= start_date) & (break_attempt_high_events_df["swing_date"] <= end_date)],
        PLOT_BREAK_ATTEMPT_MODE,
        PLOT_BREAK_MIN_CONFLUENCE,
        PLOT_BREAK_MIN_GAP_DAYS,
        REQUIRE_MULTI_HORIZON_CONFLUENCE,
        MIN_SHORT_CONFLUENCE_HITS,
        MIN_LONG_CONFLUENCE_HITS,
    )
    bl = filter_break_attempt_events_for_plot(
        break_attempt_low_events_df[(break_attempt_low_events_df["swing_date"] >= start_date) & (break_attempt_low_events_df["swing_date"] <= end_date)],
        PLOT_BREAK_ATTEMPT_MODE,
        PLOT_BREAK_MIN_CONFLUENCE,
        PLOT_BREAK_MIN_GAP_DAYS,
        REQUIRE_MULTI_HORIZON_CONFLUENCE,
        MIN_SHORT_CONFLUENCE_HITS,
        MIN_LONG_CONFLUENCE_HITS,
    )
    hh = confirmed_high_events_df[(confirmed_high_events_df["swing_date"] >= start_date) & (confirmed_high_events_df["swing_date"] <= end_date)] if not confirmed_high_events_df.empty else confirmed_high_events_df
    hl = confirmed_low_events_df[(confirmed_low_events_df["swing_date"] >= start_date) & (confirmed_low_events_df["swing_date"] <= end_date)] if not confirmed_low_events_df.empty else confirmed_low_events_df
    candidate_count = len(ch) + len(cl)
    break_count = len(bh) + len(bl)
    confirmed_count = len(hh) + len(hl)
    return {
        "start": start_date,
        "end": end_date,
        "bars": int(bars),
        "candidate_plot_count": int(candidate_count),
        "break_plot_count": int(break_count),
        "confirmed_count": int(confirmed_count),
        "candidate_per_100": float(candidate_count * 100 / bars),
        "break_per_100": float(break_count * 100 / bars),
        "confirmed_per_100": float(confirmed_count * 100 / bars),
        "noise_to_confirm_ratio": float((candidate_count + break_count) / max(1, confirmed_count)),
    }


robust_rows = []
last_750 = result_df.tail(750)
prev_750 = result_df.iloc[max(0, len(result_df) - 1500): max(0, len(result_df) - 750)]
if len(prev_750) > 0:
    m_prev = period_structure_metrics(prev_750["date"].iloc[0], prev_750["date"].iloc[-1])
    if m_prev is not None:
        m_prev["segment"] = "prev_750"
        robust_rows.append(m_prev)
m_last = period_structure_metrics(last_750["date"].iloc[0], last_750["date"].iloc[-1])
if m_last is not None:
    m_last["segment"] = "last_750"
    robust_rows.append(m_last)

for year, grp in result_df.groupby(result_df["date"].dt.year):
    if len(grp) < 120:
        continue
    m_year = period_structure_metrics(grp["date"].iloc[0], grp["date"].iloc[-1])
    if m_year is not None:
        m_year["segment"] = f"year_{year}"
        robust_rows.append(m_year)

robustness_df = pd.DataFrame(robust_rows)
if not robustness_df.empty:
    robustness_df = robustness_df.sort_values("start").reset_index(drop=True)
    cols = ["segment", "start", "end", "bars", "candidate_per_100", "break_per_100", "confirmed_per_100", "noise_to_confirm_ratio"]
    print("\nWalk-forward robustness summary:")
    display(robustness_df[cols])
    cand_cv = float(robustness_df["candidate_per_100"].std(ddof=0) / max(1e-9, robustness_df["candidate_per_100"].mean()))
    break_cv = float(robustness_df["break_per_100"].std(ddof=0) / max(1e-9, robustness_df["break_per_100"].mean()))
    conf_cv = float(robustness_df["confirmed_per_100"].std(ddof=0) / max(1e-9, robustness_df["confirmed_per_100"].mean()))
    stability = pd.DataFrame([
        {"metric": "candidate_per_100_cv", "value": cand_cv},
        {"metric": "break_per_100_cv", "value": break_cv},
        {"metric": "confirmed_per_100_cv", "value": conf_cv},
    ])
    print("Stability indicators (lower is better, ~0.35 altında genelde daha stabil):")
    display(stability)

# --- Causal Fibonacci feature generation from confirmed market structure ---
def fib_ratio_label(r):
    return f"{int(round(float(r) * 1000)):03d}"


def build_confirmed_events_for_fib(confirmed_high_df, confirmed_low_df):
    high = confirmed_high_df.copy()
    low = confirmed_low_df.copy()
    if high.empty and low.empty:
        return pd.DataFrame(columns=["kind", "available_on", "swing_date", "value", "swing_tier", "confluence_count", "confluence_windows"])
    if not high.empty:
        high["kind"] = "high"
    if not low.empty:
        low["kind"] = "low"
    events = pd.concat([high, low], ignore_index=True, sort=False)
    events = classify_confirmed_tier(events, PLOT_STRUCTURAL_MIN_TOTAL_CONFLUENCE, PLOT_STRUCTURAL_MIN_LONG_HITS, PLOT_GLOBAL_MIN_TOTAL_CONFLUENCE, PLOT_GLOBAL_MIN_LONG_HITS, PLOT_GLOBAL_REQUIRE_200_WINDOW)
    events["available_on"] = pd.to_datetime(events["available_on"])
    events["swing_date"] = pd.to_datetime(events["swing_date"])
    events = events.sort_values(["swing_date", "available_on", "kind"]).reset_index(drop=True)
    return events


def filter_fib_anchor_tier(events_df, tier_filter="structural_plus"):
    if events_df.empty:
        return events_df.copy()
    mode = str(tier_filter).lower()
    out = events_df.copy()
    if mode == "global_only":
        out = out[out["swing_tier"] == "global_extrema"]
    elif mode == "structural_plus":
        out = out[out["swing_tier"].isin(["structural_extrema", "global_extrema"])]
    return out.sort_values(["swing_date", "available_on"]).reset_index(drop=True)


def _tier_bonus(tier_name):
    t = str(tier_name)
    if t == "global_extrema":
        return 3.0
    if t == "structural_extrema":
        return 1.5
    return 0.0


def _with_structure_flags(events_df):
    if events_df.empty:
        return events_df.copy()
    events = events_df.sort_values(["swing_date", "available_on"]).copy()
    events["is_hh"] = False
    events["is_ll"] = False
    running_max_high = None
    running_min_low = None
    for idx, ev in events.iterrows():
        kind = str(ev.get("kind", ""))
        value = float(ev.get("value", np.nan))
        if not np.isfinite(value):
            continue
        if kind == "high":
            if running_max_high is not None and value >= running_max_high:
                events.at[idx, "is_hh"] = True
            running_max_high = value if running_max_high is None else max(running_max_high, value)
        elif kind == "low":
            if running_min_low is not None and value <= running_min_low:
                events.at[idx, "is_ll"] = True
            running_min_low = value if running_min_low is None else min(running_min_low, value)
    return events


def latest_opposite_leg(known_events):
    if known_events is None or len(known_events) < 2:
        return None
    e2 = known_events.iloc[-1]
    prev = known_events.iloc[:-1]
    opp = prev[prev["kind"] != e2["kind"]]
    if opp.empty:
        return None
    e1 = opp.iloc[-1]
    if e2["kind"] == "high" and e1["kind"] == "low":
        low = float(e1["value"]); high = float(e2["value"]); direction = "up"
        low_date = pd.Timestamp(e1["swing_date"]); high_date = pd.Timestamp(e2["swing_date"])
        low_tier = str(e1.get("swing_tier", "")); high_tier = str(e2.get("swing_tier", ""))
    elif e2["kind"] == "low" and e1["kind"] == "high":
        high = float(e1["value"]); low = float(e2["value"]); direction = "down"
        low_date = pd.Timestamp(e2["swing_date"]); high_date = pd.Timestamp(e1["swing_date"])
        low_tier = str(e2.get("swing_tier", "")); high_tier = str(e1.get("swing_tier", ""))
    else:
        return None
    if not np.isfinite(low) or not np.isfinite(high) or high <= low:
        return None
    pair_score = float(e1.get("confluence_count", 0) or 0) + float(e2.get("confluence_count", 0) or 0) + _tier_bonus(low_tier) + _tier_bonus(high_tier)
    return {
        "direction": direction,
        "low": low,
        "high": high,
        "range": high - low,
        "low_date": low_date,
        "high_date": high_date,
        "low_tier": low_tier,
        "high_tier": high_tier,
        "pair_score": pair_score,
        "is_hh": False,
        "is_ll": False,
    }


def highest_confluence_hh_ll_leg(known_events, lookback_events=120):
    if known_events is None or len(known_events) < 2:
        return None
    n = max(2, int(lookback_events))
    tail = known_events.sort_values(["swing_date", "available_on"]).tail(n).reset_index(drop=True)
    tail = _with_structure_flags(tail)
    highs = tail[tail["kind"] == "high"].copy()
    lows = tail[tail["kind"] == "low"].copy()
    if highs.empty or lows.empty:
        return latest_opposite_leg(tail)
    candidates = []
    for _, h in highs.iterrows():
        h_value = float(h.get("value", np.nan))
        if not np.isfinite(h_value):
            continue
        h_conf = float(h.get("confluence_count", 0) or 0)
        h_bonus = _tier_bonus(h.get("swing_tier", "")) + (1.0 if bool(h.get("is_hh", False)) else 0.0)
        h_score = h_conf + h_bonus
        h_date = pd.Timestamp(h["swing_date"])
        h_tier = str(h.get("swing_tier", ""))
        for _, l in lows.iterrows():
            l_value = float(l.get("value", np.nan))
            if not np.isfinite(l_value) or h_value <= l_value:
                continue
            l_conf = float(l.get("confluence_count", 0) or 0)
            l_bonus = _tier_bonus(l.get("swing_tier", "")) + (1.0 if bool(l.get("is_ll", False)) else 0.0)
            l_score = l_conf + l_bonus
            l_date = pd.Timestamp(l["swing_date"])
            l_tier = str(l.get("swing_tier", ""))
            pair_score = h_score + l_score
            second_date = max(h_date, l_date)
            direction = "up" if l_date <= h_date else "down"
            candidates.append({
                "direction": direction,
                "low": l_value,
                "high": h_value,
                "range": h_value - l_value,
                "low_date": l_date,
                "high_date": h_date,
                "low_tier": l_tier,
                "high_tier": h_tier,
                "pair_score": pair_score,
                "is_hh": bool(h.get("is_hh", False)),
                "is_ll": bool(l.get("is_ll", False)),
                "second_date": second_date,
            })
    if not candidates:
        return latest_opposite_leg(tail)
    candidates = sorted(candidates, key=lambda x: (x["pair_score"], x["second_date"], x["range"]), reverse=True)
    best = candidates[0]
    best.pop("second_date", None)
    return best


def build_fib_features_causal(df, fib_events, ratios, atr_window=14, min_swing_range_atr=0.8, anchor_method="hh_ll_confluence", lookback_events=120):
    work = df.copy().reset_index(drop=True)
    prev_close = work["close"].shift(1)
    prev_high = work["high"].shift(1)
    prev_low = work["low"].shift(1)
    pc = work["close"].shift(1)
    tr = pd.concat([(work["high"] - work["low"]), (work["high"] - pc).abs(), (work["low"] - pc).abs()], axis=1).max(axis=1)
    atr_prev = tr.rolling(window=int(atr_window), min_periods=1).mean().shift(1)
    events = fib_events.sort_values(["available_on", "swing_date"]).reset_index(drop=True)
    k = 0
    out_rows = []
    ratios = sorted({float(r) for r in ratios if 0.0 <= float(r) <= 1.0})
    if 0.0 not in ratios:
        ratios = [0.0] + ratios
    if 1.0 not in ratios:
        ratios = ratios + [1.0]
    mode = str(anchor_method).lower()
    for i, row in work.iterrows():
        current_date = pd.Timestamp(row["date"])
        while k < len(events) and pd.Timestamp(events.at[k, "available_on"]) < current_date:
            k += 1
        known = events.iloc[:k]
        rec = {
            "fib_leg_direction": None,
            "fib_anchor_low": np.nan,
            "fib_anchor_high": np.nan,
            "fib_anchor_range": np.nan,
            "fib_anchor_low_date": pd.NaT,
            "fib_anchor_high_date": pd.NaT,
            "fib_anchor_low_tier": None,
            "fib_anchor_high_tier": None,
            "fib_anchor_events_seen": int(len(known)),
            "fib_anchor_method": mode,
            "fib_anchor_pair_score": np.nan,
            "fib_anchor_is_hh": False,
            "fib_anchor_is_ll": False,
            "fib_0_level": np.nan,
            "fib_1_level": np.nan,
            "fib_nearest_ratio": np.nan,
            "fib_nearest_dist_prev_close_atr": np.nan,
        }
        for r in ratios:
            lbl = fib_ratio_label(r)
            rec[f"fib_{lbl}_level"] = np.nan
            rec[f"fib_{lbl}_dist_prev_close"] = np.nan
            rec[f"fib_{lbl}_dist_prev_close_atr"] = np.nan
            rec[f"fib_{lbl}_touched_prev_bar_flag"] = False
        if mode == "latest_opposite":
            leg = latest_opposite_leg(known)
        else:
            leg = highest_confluence_hh_ll_leg(known, lookback_events=lookback_events)
        if leg is not None:
            atr_v = atr_prev.iloc[i]
            min_range = 0.0 if pd.isna(atr_v) else float(min_swing_range_atr) * float(atr_v)
            if leg["range"] >= min_range:
                rec["fib_leg_direction"] = leg["direction"]
                rec["fib_anchor_low"] = leg["low"]
                rec["fib_anchor_high"] = leg["high"]
                rec["fib_anchor_range"] = leg["range"]
                rec["fib_anchor_low_date"] = leg["low_date"]
                rec["fib_anchor_high_date"] = leg["high_date"]
                rec["fib_anchor_low_tier"] = leg["low_tier"]
                rec["fib_anchor_high_tier"] = leg["high_tier"]
                rec["fib_anchor_pair_score"] = float(leg.get("pair_score", np.nan))
                rec["fib_anchor_is_hh"] = bool(leg.get("is_hh", False))
                rec["fib_anchor_is_ll"] = bool(leg.get("is_ll", False))
                nearest_ratio = np.nan
                nearest_dist_atr = np.inf
                for r in ratios:
                    lbl = fib_ratio_label(r)
                    level = leg["low"] + leg["range"] * r
                    rec[f"fib_{lbl}_level"] = float(level)
                    if abs(r - 0.0) < 1e-12:
                        rec["fib_0_level"] = float(level)
                    if abs(r - 1.0) < 1e-12:
                        rec["fib_1_level"] = float(level)
                    if not pd.isna(prev_close.iloc[i]):
                        d = float(prev_close.iloc[i] - level)
                        rec[f"fib_{lbl}_dist_prev_close"] = d
                        d_atr = np.nan if pd.isna(atr_v) or atr_v == 0 else d / float(atr_v)
                        rec[f"fib_{lbl}_dist_prev_close_atr"] = d_atr
                        if not pd.isna(d_atr) and abs(d_atr) < abs(nearest_dist_atr):
                            nearest_dist_atr = d_atr
                            nearest_ratio = r
                    if not pd.isna(prev_low.iloc[i]) and not pd.isna(prev_high.iloc[i]):
                        rec[f"fib_{lbl}_touched_prev_bar_flag"] = bool(float(prev_low.iloc[i]) <= level <= float(prev_high.iloc[i]))
                rec["fib_nearest_ratio"] = nearest_ratio
                rec["fib_nearest_dist_prev_close_atr"] = (np.nan if not np.isfinite(nearest_dist_atr) else float(nearest_dist_atr))
        out_rows.append(rec)
    return pd.DataFrame(out_rows)


fib_events_all = build_confirmed_events_for_fib(confirmed_high_events_df, confirmed_low_events_df)
fib_events = filter_fib_anchor_tier(fib_events_all, FIB_ANCHOR_TIER_FILTER)
fib_features_df = build_fib_features_causal(
    result_df,
    fib_events,
    FIB_RATIOS,
    ATR_WINDOW,
    FIB_MIN_SWING_RANGE_ATR,
    FIB_ANCHOR_METHOD,
    FIB_HH_LL_LOOKBACK_EVENTS,
)
result_with_fib = pd.concat([result_df.reset_index(drop=True), fib_features_df], axis=1)
print("\nFib anchor tier filter:", FIB_ANCHOR_TIER_FILTER)
print("Fib anchor method:", FIB_ANCHOR_METHOD, "| lookback_events:", FIB_HH_LL_LOOKBACK_EVENTS)
print("Fib events total / selected:", len(fib_events_all), len(fib_events))
print("Rows with active fib leg:", int(result_with_fib["fib_leg_direction"].notna().sum()), "of", len(result_with_fib))
fib_cols_preview = ["date", "fib_leg_direction", "fib_anchor_low", "fib_anchor_high", "fib_anchor_range", "fib_nearest_ratio", "fib_nearest_dist_prev_close_atr"]
display(result_with_fib[fib_cols_preview].tail(20))


def plot_fib_levels_on_last_bar(trailing_bars=300):
    i = len(result_with_fib) - 1
    fig = plot_until_index(i, trailing_bars=trailing_bars)
    row = result_with_fib.iloc[i]
    direction = row.get("fib_leg_direction")
    if direction is None or (isinstance(direction, float) and pd.isna(direction)):
        return fig
    plot_ratios = sorted({0.0, 1.0, *[float(r) for r in FIB_RATIOS if 0.0 <= float(r) <= 1.0]})
    for r in plot_ratios:
        lbl = fib_ratio_label(r)
        level = row.get(f"fib_{lbl}_level", np.nan)
        if pd.isna(level):
            continue
        fig.add_hline(y=float(level), line_dash="dot", line_width=1, opacity=0.45, annotation_text=f"fib {r}", annotation_position="right")
    fig.update_layout(title=f"{fig.layout.title.text} | Fib leg={direction}")
    return fig


# fib plot is executed in separate cells below


## 11A. Plot Cells (Run Separately)

In [ ]:
plot_last_n_bars(PLOT_LAST_N_BARS).show()

In [ ]:
plot_last_n_bars(len(result_df)).show()

In [ ]:
STEP_PLOT_INDEX = len(result_df) - 1
plot_until_index(STEP_PLOT_INDEX, trailing_bars=PLOT_LAST_N_BARS).show()

In [ ]:
plot_fib_levels_on_last_bar(320).show()

## 11B. Fib Anchor Audit (Causal + Update Check)


In [ ]:
def fib_anchor_audit(result_with_fib_df, fib_events_df):
    if result_with_fib_df is None or len(result_with_fib_df) == 0:
        print("No result_with_fib rows available.")
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    def _k(d, v):
        if pd.isna(d) or pd.isna(v):
            return None
        return f"{pd.Timestamp(d).strftime('%Y-%m-%d')}|{float(v):.8f}"

    audit = result_with_fib_df.copy()
    audit["fib_anchor_low_key"] = [_k(d, v) for d, v in zip(audit["fib_anchor_low_date"], audit["fib_anchor_low"])]
    audit["fib_anchor_high_key"] = [_k(d, v) for d, v in zip(audit["fib_anchor_high_date"], audit["fib_anchor_high"])]
    audit["fib_anchor_pair_key"] = audit["fib_anchor_low_key"].fillna("") + "__" + audit["fib_anchor_high_key"].fillna("")
    audit.loc[audit["fib_anchor_low_key"].isna() | audit["fib_anchor_high_key"].isna(), "fib_anchor_pair_key"] = np.nan
    audit["fib_anchor_changed_flag"] = audit["fib_anchor_pair_key"].ne(audit["fib_anchor_pair_key"].shift(1)) & audit["fib_anchor_pair_key"].notna()

    low_map, high_map = {}, {}
    if fib_events_df is not None and not fib_events_df.empty:
        ev = fib_events_df.copy()
        ev["event_key"] = [_k(d, v) for d, v in zip(ev["swing_date"], ev["value"])]
        low_map = ev[ev["kind"] == "low"].dropna(subset=["event_key"]).drop_duplicates("event_key", keep="last").set_index("event_key")["available_on"].to_dict()
        high_map = ev[ev["kind"] == "high"].dropna(subset=["event_key"]).drop_duplicates("event_key", keep="last").set_index("event_key")["available_on"].to_dict()

    audit["fib_anchor_low_available_on"] = pd.to_datetime(audit["fib_anchor_low_key"].map(low_map))
    audit["fib_anchor_high_available_on"] = pd.to_datetime(audit["fib_anchor_high_key"].map(high_map))
    audit["fib_anchor_causal_ok"] = (
        (audit["fib_anchor_low_available_on"].isna() | (audit["fib_anchor_low_available_on"] < pd.to_datetime(audit["date"])))
        & (audit["fib_anchor_high_available_on"].isna() | (audit["fib_anchor_high_available_on"] < pd.to_datetime(audit["date"])))
    )

    changes = audit[audit["fib_anchor_changed_flag"]].copy()
    active = audit[audit["fib_leg_direction"].notna()].copy()
    violations = active[~active["fib_anchor_causal_ok"]].copy()
    return audit, changes, violations


fib_anchor_audit_df, fib_anchor_change_df, fib_anchor_violations_df = fib_anchor_audit(result_with_fib, fib_events)
print("Active fib rows:", int(fib_anchor_audit_df["fib_leg_direction"].notna().sum()), "of", len(fib_anchor_audit_df))
print("Anchor update rows:", len(fib_anchor_change_df))
print("Causality violations (must be 0):", len(fib_anchor_violations_df))
display(
    fib_anchor_change_df[
        [
            "date",
            "fib_leg_direction",
            "fib_anchor_low_date",
            "fib_anchor_low",
            "fib_anchor_high_date",
            "fib_anchor_high",
            "fib_anchor_pair_score",
        ]
    ].tail(30)
)


## 11C. Fib State Visual Check (Date-Range + Snapshot)


In [ ]:
def _normalize_fib_ratio_list(ratios):
    ratio_list = sorted({float(r) for r in ratios if 0.0 <= float(r) <= 1.0})
    if 0.0 not in ratio_list:
        ratio_list = [0.0] + ratio_list
    if 1.0 not in ratio_list:
        ratio_list = ratio_list + [1.0]
    return ratio_list


def _slice_result_with_fib(start_date=None, end_date=None):
    data = result_with_fib.copy()
    data["date"] = pd.to_datetime(data["date"])
    if start_date is not None:
        data = data[data["date"] >= pd.Timestamp(start_date)]
    if end_date is not None:
        data = data[data["date"] <= pd.Timestamp(end_date)]
    if data.empty:
        raise ValueError("Selected date window has no rows.")
    return data


def _fib_anchor_pair_key(low_date, low_val, high_date, high_val):
    if pd.isna(low_date) or pd.isna(low_val) or pd.isna(high_date) or pd.isna(high_val):
        return np.nan
    return (
        f"{pd.Timestamp(low_date).strftime('%Y-%m-%d')}|{float(low_val):.8f}"
        f"__{pd.Timestamp(high_date).strftime('%Y-%m-%d')}|{float(high_val):.8f}"
    )


def plot_fib_levels_evolution(start_date=None, end_date=None, ratios=(0.0, 0.382, 0.5, 0.618, 1.0), show_candles=True, show_anchor_change_markers=True):
    data = _slice_result_with_fib(start_date, end_date)
    ratio_list = _normalize_fib_ratio_list(ratios)

    fig = go.Figure()
    if bool(show_candles):
        fig.add_trace(go.Candlestick(
            x=data["date"], open=data["open"], high=data["high"], low=data["low"], close=data["close"], name="Price"
        ))
    else:
        fig.add_trace(go.Scatter(x=data["date"], y=data["close"], mode="lines", name="Close", line=dict(color="#2a9d8f", width=1.6)))

    active = data[data["fib_leg_direction"].notna()].copy()
    if not active.empty:
        fig.add_trace(go.Scatter(
            x=active["date"], y=active["fib_anchor_low"], mode="lines", line_shape="hv",
            name="Fib Anchor Low", line=dict(color="#1d3557", width=1.9)
        ))
        fig.add_trace(go.Scatter(
            x=active["date"], y=active["fib_anchor_high"], mode="lines", line_shape="hv",
            name="Fib Anchor High", line=dict(color="#800f2f", width=1.9)
        ))

        palette = {
            0.0: "#343a40",
            0.236: "#6c757d",
            0.382: "#6c757d",
            0.5: "#495057",
            0.618: "#6c757d",
            0.786: "#6c757d",
            1.0: "#343a40",
        }
        for r in ratio_list:
            lbl = fib_ratio_label(r)
            col = f"fib_{lbl}_level"
            if col not in active.columns:
                continue
            fig.add_trace(go.Scatter(
                x=active["date"], y=active[col], mode="lines", line_shape="hv",
                name=f"Fib {r:.3f}",
                line=dict(color=palette.get(r, "#6c757d"), width=1.6 if r in (0.0, 1.0, 0.5) else 1.0, dash="dot"),
                opacity=0.78 if r in (0.0, 1.0, 0.5) else 0.48,
            ))

        if bool(show_anchor_change_markers):
            pair_keys = [
                _fib_anchor_pair_key(ld, lv, hd, hv)
                for ld, lv, hd, hv in zip(
                    active["fib_anchor_low_date"], active["fib_anchor_low"], active["fib_anchor_high_date"], active["fib_anchor_high"]
                )
            ]
            active = active.copy()
            active["anchor_pair_key"] = pair_keys
            change_mask = active["anchor_pair_key"].ne(active["anchor_pair_key"].shift(1)) & active["anchor_pair_key"].notna()
            changes = active[change_mask]
            if not changes.empty:
                fig.add_trace(go.Scatter(
                    x=changes["date"],
                    y=changes["close"],
                    mode="markers",
                    name="Anchor Updated",
                    marker=dict(symbol="diamond-open", size=9, color="#ff7f11"),
                    hovertemplate="Anchor updated<br>Date=%{x}<br>Close=%{y:.2f}<extra></extra>",
                ))

    title = f"Fib Level Evolution ({data['date'].min().date()} to {data['date'].max().date()})"
    fig.update_layout(
        title=title,
        template="plotly_white",
        xaxis_title="Date",
        yaxis_title="Price",
        xaxis_rangeslider_visible=False,
        height=760,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0),
    )
    return fig


def plot_fib_state_at_date(target_date, trailing_bars=320, ratios=(0.0, 0.382, 0.5, 0.618, 1.0)):
    ts = pd.Timestamp(target_date)
    data = result_with_fib.copy()
    data["date"] = pd.to_datetime(data["date"])
    valid = data[data["date"] <= ts]
    if valid.empty:
        raise ValueError(f"No bars found on or before {ts.date()}.")

    i = int(valid.index[-1])
    row = data.loc[i]
    fig = plot_until_index(i, trailing_bars=trailing_bars)

    direction = row.get("fib_leg_direction")
    if pd.isna(direction):
        fig.update_layout(title=f"Fib snapshot at {row['date'].date()} | No active fib anchors yet")
        return fig

    ratio_list = _normalize_fib_ratio_list(ratios)
    for r in ratio_list:
        lbl = fib_ratio_label(r)
        level_col = f"fib_{lbl}_level"
        level = row.get(level_col, np.nan)
        if pd.isna(level):
            continue
        if r in (0.0, 1.0):
            color, width, opacity = ("#343a40", 1.8, 0.72)
        elif r == 0.5:
            color, width, opacity = ("#495057", 1.4, 0.62)
        else:
            color, width, opacity = ("#6c757d", 1.0, 0.42)
        fig.add_hline(
            y=float(level),
            line_dash="dot",
            line_color=color,
            line_width=width,
            opacity=opacity,
            annotation_text=f"fib {r:.3f}",
            annotation_position="right",
        )

    low_date = pd.Timestamp(row.get("fib_anchor_low_date")).date() if pd.notna(row.get("fib_anchor_low_date")) else None
    high_date = pd.Timestamp(row.get("fib_anchor_high_date")).date() if pd.notna(row.get("fib_anchor_high_date")) else None
    fig.update_layout(
        title=(
            f"Fib snapshot at {row['date'].date()} (causal) | leg={direction}"
            f" | low={row.get('fib_anchor_low'):.2f} @ {low_date}"
            f" | high={row.get('fib_anchor_high'):.2f} @ {high_date}"
        )
    )
    return fig


In [ ]:
FIB_VIEW_START = "2020-01-01"
FIB_VIEW_END = "2021-06-30"
plot_fib_levels_evolution(
    start_date=FIB_VIEW_START,
    end_date=FIB_VIEW_END,
    ratios=(0.0, 0.382, 0.5, 0.618, 1.0),
    show_candles=True,
    show_anchor_change_markers=True,
).show()


In [ ]:
FIB_INSPECT_DATE = "2020-11-20"
plot_fib_state_at_date(
    target_date=FIB_INSPECT_DATE,
    trailing_bars=320,
    ratios=(0.0, 0.236, 0.382, 0.5, 0.618, 0.786, 1.0),
).show()
